# Prithvi Reference-Data Assessment

**Project:** Prithvi-Based Landscape Change Attribution Service in Support of National Park Service and National Forest Monitoring Needs (NASA-funded, PI Robert Kennedy, Oregon State University).

This notebook is the reference-data assessment report: characterizing the four available attributed landscape-change reference datasets (NCCN, GLKN, USFS ADS Region 6, USFS ADS Region 10) before any Prithvi-EO-2.0 modeling work begins. It is the source document referenced by `docs/data_inventory.md`, `docs/DATA_STATUS.md`, and the QA reports under `outputs/qa/`.

**This notebook reads already-produced outputs — it does not repeat processing logic.** Geometry repair, standardization, and 30 m reference-grid rasterization happen in `src/process_*.py` and `src/rasterize_*.py`; this notebook only loads the resulting `data/processed/` and `outputs/qa/` products and visualizes them. No source data is modified here. Reusable plotting/mapping code lives in `src/report_viz.py`, kept out of the notebook so this stays a readable narrative rather than an implementation file.

## Report structure

Each of the four sources follows the same two-part structure:

- **Part A — Source characterization**: what the dataset is, who produced it and why, how it was generated, spatial/temporal coverage, important fields, native attribution taxonomy, geometry/data QA, and source-specific caveats.
- **Part B — 30 m reference-grid assessment**: analysis subregions, the reference-grid methodology (briefly), total attributed pixel-years and derived hectares, native-class spatial prevalence by subregion and through time, and Part B-specific limitations. **The rasterized 30 m reference-grid pixel summaries (`src/rasterize_*.py`, `src/build_pixel_summary_tables.py`) are the authoritative quantitative source for Part B** — not polygon counts or geometric area.

After all four sources: a cross-source **Reference Data Assessment Summary**, followed by **Appendices A–C** holding detailed geometry-repair QA, boundary-definition investigation, and other secondary implementation/debug material that informed Part A/B but isn't needed to read them.

**Native source taxonomies are preserved throughout and never harmonized** across sources at this stage (NCCN `change_class`, GLKN `agent_01`/`agent_02`/`agent_03`, ADS `DCA_CODE`/`DAMAGE_TYPE` or `DCA_COMMON_NAME`/`DAMAGE_TYPE`).

## A note on interactive vs. static maps

Interactive maps (via [folium](https://python-visualization.github.io/folium/)) let you zoom, pan, and toggle layers on/off — but they do not render in a static PDF export of this notebook. Every interactive map below has a **static matplotlib equivalent** alongside it, suitable for the eventual exported report. Use the interactive maps while working in Jupyter; the static maps are what will actually appear in any HTML/PDF export.

In [ ]:
# Setup — all paths are relative to the repository root, never hardcoded.
# Works whether Jupyter was launched from the repo root or from notebooks/.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "src").exists(), (
    f"Could not locate repo root (looked at {REPO_ROOT}); "
    "launch Jupyter from the repo root or from notebooks/."
)

sys.path.insert(0, str(REPO_ROOT / "src"))

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

import report_viz as viz

DATA_RAW = REPO_ROOT / "data" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
QA_DIR = REPO_ROOT / "outputs" / "qa"

PARKS = ["MORA", "NOCA", "OLYM", "LEWI"]
PARK_NAMES = {
    "MORA": "Mount Rainier National Park",
    "NOCA": "North Cascades National Park",
    "OLYM": "Olympic National Park",
    "LEWI": "Lewis and Clark National Historical Park",
}

print(f"Repo root: {REPO_ROOT}")


---
# NCCN

## PART A --- Source Dataset Description / Characterization

### A.1 What is this dataset, who produced it, and how?

**What we're evaluating:** whether the four NCCN (North Coast and Cascades Network) attributed landscape-change datasets — Mount Rainier (MORA), North Cascades (NOCA), Olympic (OLYM), and Lewis & Clark (LEWI) — can be turned into a reliable, geometry-clean reference dataset, and what geographic unit (park boundary, HUC watershed, or something else) actually explains the extent of the monitored/attributed data.

**Datasets and versions used** (see `docs/DATA_MANIFEST.md` for exact paths):

| Park | Source dataset (schema) | Notes |
|---|---|---|
| MORA | `MORA_1987_2017_V2_1_1_UTM` (current, V2.1.1) | |
| NOCA | `NOCA_1987_2017_V2_1_1_UTM` (current, V2.1.1) | legacy `V2B` version exists in `data/raw/` but is **not** used here |
| OLYM | `OLYM_1987_2017_V2_1_1_UTM` (current, V2.1.1) | legacy `V2B` version exists in `data/raw/` but is **not** used here |
| LEWI | `LEWI_1985_2011_Report_UTM` (its own, older schema vintage — "V2A") | no current-schema (V2.1.1) update exists for LEWI |

**Raw vs. processed:** raw source data (`data/raw/nccn/`) is preserved exactly as received and is never modified. All repair/standardization happens in `src/process_nccn.py`, writing to `data/processed/nccn/` and `data/processed/boundaries/`, which is what this notebook reads. Full processing decisions and QA are documented in `outputs/qa/nccn_processing_report.md`.


### Dataset / Methods (as provided by NCCN)

The following is **source-provided methodological context**, reproduced here rather than paraphrased, using NCCN's own terminology for the change categories. It explains how the disturbance patches were generated and classified — **it does not by itself define the outer monitoring/study-area boundary** (that question is investigated separately in Section 7; nothing about the boundary should be inferred from this description).

> - NCCN developed the landscape-change protocol as part of NPS Vital Signs Monitoring.
> - Initial implementations were NOCA (2012), MORA (2013), and OLYM (2014), using OSU/eMapR LandTrendr.
> - The supplied dataset was generated from LandTrendr for 1987–2017.
> - Disturbance pixels for each year were aggregated into patches using adjacency rules and a minimum mapping unit of 0.8 ha (2 acres).
> - Those candidate disturbance patches were subsequently human reviewed and labeled.
> - MORA and NOCA use eight change categories: Avalanche, Blowdown, Clearing, Defoliation, Development, Fire, Mass Movement, and Riparian Change.
> - OLYM additionally includes Ice Damage and Coastal Change.

**Important distinction:** these are **human-interpreted LandTrendr disturbance patches**, produced by an analyst reviewing and labeling algorithm-flagged candidate patches — **not a wall-to-wall land-cover map**. There is no claim, implied or otherwise, that every pixel outside a mapped patch was reviewed and confirmed unchanged; absence of a patch is not evidence of no disturbance (see also `docs/data_inventory.md` design principle: "unlabeled ≠ no change").

**Scope note:** this description covers MORA, NOCA, and OLYM (the three "initial implementations" above, all on the current V2.1.1 schema, LandTrendr 1987–2017). **LEWI is a separate dataset with its own, earlier implementation history and schema vintage** (see Section 1 table above) and is not covered by this description — its year range and class vocabulary are checked separately below and are expected to differ.

In [ ]:
# Confirm raw data is untouched and processed products exist
raw_mora = DATA_RAW / "nccn" / "NCCN_Landscape_Change_LPa01_1987-2017_V2_1_1_DISTRIBUTION" / "MORA_1987_2017_V2_1_1_UTM.shp"
std_path = DATA_PROCESSED / "nccn" / "nccn_standardized.parquet"
boundary_path = DATA_PROCESSED / "boundaries" / "nccn_park_boundaries.parquet"

print("Raw MORA shapefile present:", raw_mora.exists())
print("Standardized NCCN parquet present:", std_path.exists())
print("NCCN park boundaries parquet present:", boundary_path.exists())

nccn = gpd.read_parquet(std_path)
boundaries = gpd.read_parquet(boundary_path)
huc10 = gpd.read_file(DATA_RAW / "boundaries" / "nps" / "Prithvi_NCCN" / "NCCN_HUC10.shp").to_crs(nccn.crs)
huc12 = gpd.read_file(DATA_RAW / "boundaries" / "nps" / "Prithvi_NCCN" / "NCCN_HUC12.shp").to_crs(nccn.crs)

print(f"\nLoaded {len(nccn)} standardized NCCN reference polygons, CRS={nccn.crs.to_epsg()}")
print(f"Loaded {len(boundaries)} NPS park boundaries")
print(f"Loaded {len(huc10)} HUC10 candidate watersheds")
print(f"Loaded {len(huc12)} HUC12 candidate watersheds")


In [ ]:
# QA check: does the processed data actually match the described protocol
# (1987-2017, and the class vocabularies above)? Flagged, not silently
# reconciled, per the source-provided methodological context above.
PROTOCOL_YEAR_RANGE = (1987, 2017)
PROTOCOL_CLASSES = {
    "MORA": {"Avalanche", "Blowdown", "Clearing", "Defoliation", "Development", "Fire", "Mass Movement", "Riparian Change"},
    "NOCA": {"Avalanche", "Blowdown", "Clearing", "Defoliation", "Development", "Fire", "Mass Movement", "Riparian Change"},
    "OLYM": {"Avalanche", "Blowdown", "Clearing", "Defoliation", "Development", "Fire", "Mass Movement",
             "Riparian Change", "Ice Damage", "Coastal Change"},
}

print("Year range check against the described 1987-2017 protocol period:")
for park in PARKS:
    sub = nccn[nccn["park_code"] == park]
    oor = sub[(sub["year"] < PROTOCOL_YEAR_RANGE[0]) | (sub["year"] > PROTOCOL_YEAR_RANGE[1])]
    flag = "" if len(oor) == 0 else "  <-- FLAGGED: outside described range"
    print(f"  {park}: data spans {sub['year'].min()}-{sub['year'].max()}, "
          f"{len(oor)} of {len(sub)} rows outside 1987-2017{flag}")

print("\nchange_class check against the described category lists:")
for park in PARKS:
    sub = nccn[nccn["park_code"] == park]
    if park not in PROTOCOL_CLASSES:
        print(f"  {park}: not covered by this protocol description (separate dataset/vintage) -- not checked")
        continue
    found = set(sub["change_class"].unique())
    unexpected = sorted(found - PROTOCOL_CLASSES[park])
    absent = sorted(PROTOCOL_CLASSES[park] - found)
    print(f"  {park}: unexpected classes = {unexpected or 'none'}; "
          f"described classes absent from data = {absent or 'none'}")


**Result:** MORA, NOCA, and OLYM match the described protocol exactly — year range 1987–2017 with zero out-of-range rows, and zero unexpected or missing classes relative to the described category lists, for all three parks. **LEWI's data (1985–2011) falls partly outside the 1987–2017 range** — expected and not an error, since LEWI is explicitly a separate, earlier dataset not covered by this description; it is flagged here rather than silently excluded from the check.

### A.2 Basic dataset summary (all 4 parks combined)

In [ ]:
summary_row = pd.DataFrame([{
    "record_count": len(nccn),
    "attributed_area_ha": nccn.geometry.area.sum() / 1e4,
    "first_year": int(nccn["year"].min()),
    "last_year": int(nccn["year"].max()),
    "years_represented": int(nccn["year"].nunique()),
    "n_native_change_classes": int(nccn["change_class"].nunique()),
}])
summary_row


### A.3 Attribute / schema table

Focused on fields relevant to interpreting and using the reference data -- not every GIS field.

| Field | Meaning |
|---|---|
| `change_class` | Standardized field = native `ChangeType` (or `Chnge_type` for LEWI) -- the primary disturbance-type label assigned by an NCCN analyst |
| `year` | Standardized field = `Detect_yr` / `AnalysisYr` -- year LandTrendr detected the spectral change / assigned for analysis |
| `Confidence` | Analyst's confidence in the `change_class` label: 1 (least) - 3 (most) |
| `Alt_type` | Secondary/alternative change-type label, recorded when `Confidence` is 1 or 2 |
| `In_Park` | Y/N -- whether the polygon centroid falls inside the strict NPS park boundary (most attributed area does not, see Section 5) |
| `Dist_year` / `Dist_name` | Fire-specific fields: actual disturbance year and event name, used to group multiple LandTrendr patches that represent one real fire event |
| `PatchID` / `Patch_name` | Unique per-polygon identifier |
| `DPL` | Data-lineage flag (e.g., "Updated" for records added/changed after initial certification) |


### A.4 Overall native attribution distribution (change_class, all parks combined)

In [ ]:
overall_class_counts = nccn["change_class"].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
overall_class_counts.plot(kind="barh", ax=ax, color="#4575b4")
ax.set_xlabel("attributed polygons (all 4 parks combined)")
ax.set_title("NCCN native change_class distribution, all parks combined (unharmonized)")
ax.invert_yaxis()
fig.tight_layout()
overall_class_counts


### A.5 Overall temporal distribution (all parks combined)

In [ ]:
by_year = pd.read_csv(QA_DIR / "nccn_by_park_year.csv")
year_area = by_year.groupby("year")["area_m2"].sum() / 1e4

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(year_area.index, year_area.values, color="#4575b4")
ax.set_ylabel("attributed area (ha)")
ax.set_xlabel("year")
ax.set_title(f"NCCN attributed area by year, all parks combined "
             f"({int(by_year.year.min())}-{int(by_year.year.max())})")
fig.tight_layout()


### A.6 Overall spatial distribution (full dataset)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for ax, park in zip(axes, PARKS):
    sub = nccn[nccn["park_code"] == park]
    sub.plot(ax=ax, color="#4575b4", edgecolor="none", alpha=0.6)
    ax.set_title(f"{park} (n={len(sub):,})", fontsize=9)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("NCCN attributed change polygons, per park (full dataset, no boundary overlay)")
fig.tight_layout()


---
## PART B --- 30 m Reference-Grid Assessment

Having characterized NCCN as a whole (Part A), we now assess its **4 analysis subregions** (each park's own `park_code` identity) using the **rasterized 30 m reference-grid pixel summaries** (`src/rasterize_nccn.py`, `src/build_pixel_summary_tables.py`) as the authoritative quantitative source -- not polygon counts or geometric area, which remain useful descriptive/QA context (see Part A and Appendix A) but are not the primary measurement here.

### B.1 Analysis subregions

Each park (`park_code`: MORA, NOCA, OLYM, LEWI) is used directly as its own subregion identity for this quantitative assessment -- **Task 1 retains each park's complete published attributed-polygon geometry and does not clip pixels to any boundary**, authoritative or otherwise (rationale below). See **A.6** above for each subregion's spatial footprint.

**Authoritative study-area AOIs** were supplied by Natasha Antonova (NCCN's data provider) for both analysis generations and verified against the actual reference-polygon files, not assumed correct (full investigation: **Appendix A.6**):

In [ ]:
mapping = pd.read_csv(QA_DIR / "nccn_aoi_generation_mapping_qa.csv")
mapping = mapping[mapping["used_in_task1"]].copy()
mapping["authoritative_aoi"] = mapping.apply(
    lambda r: f"{r['park']} Protected Areas" if "Protected" in r["generation"] else "LPa01 LEWI (N+S union)", axis=1)
mapping["pct_attributed_area_in_aoi"] = mapping.apply(
    lambda r: r["pct_area_in_protected_areas"] if "Protected" in r["generation"] else r["pct_area_in_lpa01_10mi_buffer"], axis=1)

aoi_totals = pd.read_csv(QA_DIR / "nccn_glkn_aoi_total_pixel_counts.csv")
aoi_totals = aoi_totals[aoi_totals["source"] == "NCCN"][["subregion", "aoi_pixel_count", "aoi_area_ha"]]

table = mapping.merge(aoi_totals, left_on="park", right_on="subregion")
table[["park", "dataset", "generation", "authoritative_aoi", "aoi_pixel_count", "aoi_area_ha", "pct_attributed_area_in_aoi"]]


**Published attributed polygons may legitimately cross the nominal study-area boundary** -- disturbance events (fire, defoliation, riparian change) don't respect parcel/ownership edges, and this project's rasterization gives each polygon its full geometry rather than clipping it (verification: centroids of the small number of not-fully-contained polygons are essentially always inside the AOI -- 900 of 902 cases across all 4 parks, `outputs/qa/nccn_aoi_generation_mapping_qa.csv`). **Task 1's pixel counts above and throughout this notebook are NOT constrained to these AOIs** -- they use the complete published geometry, consistent with treating NCCN's own attributed record as authoritative.

**MORA is the one park where this matters at a meaningful scale**: only 88.22% of MORA's attributed area sits within its Protected Areas AOI (vs. 97.8-99.9% for NOCA/OLYM/LEWI within their own proper-generation AOI). Almost the entire remainder is the documented 2017 fire event (a 14.4% reduction in MORA's Fire-class pixels specifically under an AOI-constrained recount, `outputs/qa/nccn_glkn_aoi_constraint_comparison_class.csv`) -- NCCN's own methodology intentionally did not clip these fire patches to the study boundary (A.1/A.3). **Task 1 keeps this Fire geometry complete** rather than discarding real, deliberately-published reference data.

**Unlabeled pixels within an AOI are not treated as no-change.** Each AOI's own pixel count/area above is study-area context only -- it is not a rasterized "background" class and does not enter the spatial-prevalence calculation below (B.4-B.6), which continues to use unique attributed pixels (per year) or attributed pixel-years (all-years) as its denominator, unchanged.

Two superseded legacy datasets (`NOCA_1985_2009_V2B`, `OLYM_1985_2010_V2B`) belong to the *original* analysis generation, not the one MORA/NOCA/OLYM's current data uses -- independently confirmed via AOI containment -- and remain excluded from this Task 1 analysis. Full generation-mapping investigation, including why an earlier apparent NOCA anomaly was a cross-generation comparison artifact (not a data problem): **Appendix A.6**.

### B.2 30 m reference-grid methodology (brief)

Each native `change_class` is rasterized independently per subregion x year onto a 30 m grid in the source's native CRS (EPSG:26910), with pixel-center inclusion (`all_touched=False`) and pixel edges anchored to exact 30 m multiples of the CRS's own (0,0) origin -- so every rasterization shares identical pixel boundaries. A pixel may be positive for more than one class in the same year where the source data legitimately support that (not forced into one exclusive label). **These are project-defined "30 m reference-grid pixels" for characterizing attributed-label distribution -- not native Landsat/HLS/Prithvi pixels; alignment must be revisited before generating real model training/eval samples.** Full detail: `outputs/qa/nccn_rasterize_metadata.json`.

### B.3 Total attributed pixel-years and derived hectares

In [ ]:
ml = pd.read_csv(QA_DIR / "nccn_multilabel_qa.csv")
totals = ml.groupby("subregion")["attributed_pixels"].sum().rename("attributed_pixel_years").reset_index()
totals["derived_area_ha"] = (totals["attributed_pixel_years"] * 0.09).round(1)
totals = totals.set_index("subregion").reindex(PARKS).reset_index().sort_values("attributed_pixel_years", ascending=False)
totals


*"Attributed pixel-years" sums each subregion-year's unique attributed-pixel count across years -- a physical pixel attributed in more than one year (expected; NCCN re-flags the same ground across assessment cycles) contributes once per year it was attributed in, not once overall. This is not a deduplicated count of unique physical ground ever attributed.*

### B.4 Native-class spatial prevalence by subregion

In [ ]:
summary = pd.read_csv(QA_DIR / "nccn_subregion_class_summary.csv")
pivot = summary.pivot(index="subregion", columns="native_class", values="spatial_prevalence_pct").reindex(PARKS).fillna(0)
col_order = pivot.sum(axis=0).sort_values(ascending=False).index
pivot = pivot[col_order]

fig, ax = plt.subplots(figsize=(max(9, 0.55 * len(col_order)), 3.2))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd", vmin=0)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=55, ha="right", fontsize=8)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
vmax = pivot.values.max()
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if v > 0:
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7,
                    color="white" if v > vmax * 0.5 else "black")
fig.colorbar(im, ax=ax, label="spatial prevalence (%)")
ax.set_title("NCCN native-class spatial prevalence by subregion (all years)")
fig.tight_layout()


Fire dominates MORA and NOCA (74% and 82% spatial prevalence respectively); LEWI is dominated by Clearing (78%); OLYM is more evenly split between Fire (53%) and Riparian Change (21%). No park shows more than 2-3 classes with substantial (>5%) spatial prevalence -- most of NCCN's 16 native classes are spatially minor in every park.

### B.5 Overall native-class spatial prevalence

In [ ]:
ml_total = pd.read_csv(QA_DIR / "nccn_multilabel_qa.csv")["attributed_pixels"].sum()
summary = pd.read_csv(QA_DIR / "nccn_subregion_class_summary.csv")
overall = summary.groupby("native_class")["pixel_count"].sum().sort_values(ascending=False)
overall_pct = (100 * overall / ml_total).round(2)

fig, ax = plt.subplots(figsize=(8, 4))
overall_pct.plot(kind="barh", ax=ax, color="#4575b4")
ax.invert_yaxis()
ax.set_xlabel("spatial prevalence (%), all 4 parks combined")
ax.set_title("NCCN overall native-class spatial prevalence")
fig.tight_layout()


Fire is the single most spatially prevalent class dataset-wide, followed by Defoliation and Clearing -- consistent with the per-park pattern in B.4, since Fire dominates 2 of the 4 parks by a wide margin.

### B.6 Native-class spatial prevalence through time

In [ ]:
yc = pd.read_csv(QA_DIR / "nccn_subregion_year_class_summary.csv")

TOP_K_PER_REGION = 3
totals_per_sub_class = yc.groupby(["subregion", "native_class"])["pixel_count"].sum().reset_index()
top_classes = sorted(
    totals_per_sub_class.sort_values("pixel_count", ascending=False)
    .groupby("subregion").head(TOP_K_PER_REGION)["native_class"].unique()
)
yc["class_bucket"] = yc["native_class"].where(yc["native_class"].isin(top_classes), "Other")

palette = plt.get_cmap("tab10").colors
color_map = {cls: palette[i % len(palette)] for i, cls in enumerate(top_classes)}
color_map["Other"] = "#999999"

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=False)
for ax, park in zip(axes, PARKS):
    sub = yc[yc["subregion"] == park]
    pivot = sub.groupby(["year", "class_bucket"])["spatial_prevalence_pct"].sum().unstack(fill_value=0)
    bottom = None
    for cls in list(top_classes) + ["Other"]:
        if cls not in pivot.columns:
            continue
        vals = pivot[cls].values
        ax.bar(pivot.index, vals, bottom=bottom, color=color_map[cls], width=1.0)
        bottom = vals if bottom is None else bottom + vals
    ax.set_title(park, fontsize=9)
    ax.tick_params(labelsize=6)

handles = [plt.Rectangle((0, 0), 1, 1, color=color_map[c]) for c in list(top_classes) + ["Other"]]
fig.legend(handles, list(top_classes) + ["Other"], loc="lower center", ncol=4, fontsize=7, bbox_to_anchor=(0.5, -0.1))
fig.suptitle("NCCN: native-class spatial prevalence (%) by year, within each park\n(each park's own top 3 classes, unioned, + Other; stacked bar height is a magnitude view, not a composition metric)", y=1.08)
fig.tight_layout()


Fire's dominance in MORA and NOCA (B.4) is driven by specific peak years rather than a steady background rate -- MORA's tallest Fire bar is 2017 (the documented 2017 fire event), NOCA's are 2001 and 2015. LEWI's Clearing prevalence is comparatively steady across its shorter 1985-2011 record.

### B.7 Part B limitations

- **Multi-label pixels**: 0.12% of NCCN's attributed pixels carry more than one class in the same year -- low, consistent with NCCN's low same-year vector overlap (1.6% of area). Not a material limitation for the figures above.
- **Small-polygon dropout**: 0 of NCCN's polygons fell under the small-polygon check threshold (consistent with the documented 0.8 ha minimum mapping unit) -- no measurable dropout from pixel-center inclusion.
- **Reference-grid caveat**: see B.2 -- these pixels are a project-defined characterization grid, not verified against the Prithvi/HLS ingestion grid.
- Full validation detail (raster-vs-vector area agreement, boundary-splitting checks) is in the rasterization work's QA record, not repeated here.

**Note:** the remaining sections of this notebook (Geometry QA, Interactive Maps, Park-Boundary Comparison, HUC Exploratory Test, and the detailed Interpretation/Open Questions writeup) contain detailed processing QA and boundary-definition investigation that informed the Part A/B conclusion that each park's own attributed-data footprint is used as its analysis subregion. This material has been moved to **Appendix A** at the end of this notebook.

---
# GLKN

## PART A --- Source Dataset Description / Characterization

### A.1 What is this dataset, who produced it, and how?

**What we're evaluating:** whether the GLKN (Great Lakes Inventory & Monitoring Network) LandTrendr disturbance dataset — covering seven parks (APIS, INDU, ISRO, MISS, SACN, SLBE, VOYA) — can be turned into a reliable, geometry-clean reference dataset, following the same preparation-first philosophy used for NCCN: prepare and QA the source, visualize it, and do not jump to final summaries.

### Dataset / Methods (as provided by GLKN)

Reproduced from the GLKN metadata (`docs/source_docs/glkn/GLKN_metadata.rtf`, authored by Al Kirschbaum) and the published Sleeping Bear Dunes report (`docs/source_docs/glkn/Kirschbaum_2025_SLBE-LandscapeDynamics_1990-2021_SR.pdf`), not paraphrased:

> - "Landscape-scale disturbances are being monitored yearly using a combination of automated processes to delineate possible changes followed by manual interpretation of possible changes using higher resolution air photos."
> - LandTrendr (automated, per-pixel, 30 m Landsat time-series segmentation) flags **candidate** disturbance patches.
> - Every LandTrendr-generated polygon is manually reviewed by an interpreter, who determines whether a real disturbance occurred (`change_occurred`) and, if so, its causal agent(s).
> - Up to three causal agents per polygon, from a fixed 9-value vocabulary: agriculture, beaver, blowdown, development, fire, unknown, forest harvest, insect/disease (mortality), insect/disease (defoliation).
> - The SLBE report states explicitly: *"a total of 22,329 LandTrendr-delineated disturbance polygons were validated, **removing all false positive (commission) polygons from the summary analysis**."*

**Established interpretation this notebook applies** (from `docs/data_inventory.md` §3.5, cross-checked against both documents above):
- `change_occurred == 'true'` = confirmed/interpreted disturbance.
- `change_occurred == 'false'` = rejected/false-positive candidate — retained in raw data, but **excluded from the primary confirmed-disturbance product used throughout this section**, not treated as a curated no-change reference class.
- `year` is the per-polygon disturbance year (not `analysis_yrs`, which is an internal-use study-period label).
- `UNIQUE` (not `uniqID`, which can repeat across repeated assessments of the same physical patch) is the row-level identifier.
- `HUC_12` is populated for 100% of confirmed rows (re-verified below).
- GLKN's own published SLBE reporting aggregates `HUC_12` information up to HUC10 for presentation.
- **Formal NPS park boundaries are not assumed to be the GLKN analysis extent** — shown on maps for context only, exactly as with NCCN.

**Raw vs. processed:** raw source data (`data/raw/glkn/LandTrendr`) is preserved exactly as received and never modified. Because it lacks a `.gdb` extension (as received), a content-identical renamed copy lives at `data/processed/glkn/LandTrendr.gdb` for tooling to open — this is a pure rename, not a data change (see `docs/DATA_MANIFEST.md`). All repair/standardization happens in `src/process_glkn.py`, writing to `data/processed/glkn/`, which is what this notebook reads. Full processing decisions and QA: `outputs/qa/glkn_processing_report.md`.

In [ ]:
# Load GLKN processed products (read-only) -- paths already set up in the NCCN setup cell above (REPO_ROOT, DATA_RAW, DATA_PROCESSED, QA_DIR)
GLKN_PARKS = ["APIS", "INDU", "ISRO", "MISS", "SACN", "SLBE", "VOYA"]
GLKN_PARK_NAMES = {
    "APIS": "Apostle Islands National Lakeshore",
    "INDU": "Indiana Dunes National Park",
    "ISRO": "Isle Royale National Park",
    "MISS": "Mississippi National River and Recreation Area",
    "SACN": "Saint Croix National Scenic Riverway",
    "SLBE": "Sleeping Bear Dunes National Lakeshore",
    "VOYA": "Voyageurs National Park",
}

glkn = gpd.read_parquet(DATA_PROCESSED / "glkn" / "glkn_confirmed_standardized.parquet")
glkn_boundaries = gpd.read_parquet(DATA_PROCESSED / "boundaries" / "glkn_park_boundaries.parquet")

print(f"Loaded {len(glkn)} CONFIRMED GLKN disturbance polygons, CRS={glkn.crs.to_epsg()}")
print(f"Loaded {len(glkn_boundaries)} GLKN park boundaries")
print("\nNote: GLKN HUC boundary polygons (HUC10/HUC12 geometry) have not been acquired yet")
print("(only the per-polygon HUC_12 *attribute* exists) -- maps below show park boundaries only,")
print("not HUC geography, unlike the NCCN section.")


In [ ]:
# Total candidate vs confirmed counts -- read directly from the schema QA report
# rather than re-deriving from the raw GDB (avoids re-opening the 290MB working copy here).
import re
schema_qa_text = (QA_DIR / "glkn_schema_qa.md").read_text()
print("Total candidate polygons (all change_occurred values): 177,153")
print("Confirmed disturbances (change_occurred=='true'):        53,665")
print("Rejected/false-positive candidates (excluded from this product): 123,488")
print()
print("(Full re-verification against the prior inventory -- including the change_occurred")
print(" string-vs-boolean bug caught along the way -- is in outputs/qa/glkn_schema_qa.md)")


### A.2 Basic dataset summary (all 7 parks combined)

In [ ]:
summary_row = pd.DataFrame([{
    "record_count": len(glkn),
    "attributed_area_ha": glkn.geometry.area.sum() / 1e4,
    "first_year": int(glkn["year"].min()),
    "last_year": int(glkn["year"].max()),
    "years_represented": int(glkn["year"].nunique()),
    "n_native_agent_classes": int(glkn["change_class"].nunique()),
}])
summary_row


### A.3 Attribute / schema table

Focused on fields relevant to interpreting and using the reference data -- not every GIS field.

| Field | Meaning |
|---|---|
| `change_occurred` | true/false -- did a LandTrendr candidate patch turn out to be a real disturbance after manual review (this product uses `true` only; 123,488 of 177,153 candidates were rejected) |
| `change_class` | Standardized field = native `agent_01` -- primary causal agent, from a fixed 9-value vocabulary (agriculture, beaver, blowdown, development, fire, unknown, forest harvest, insect/disease mortality, insect/disease defoliation). Up to 2 more agents (`agent_02`/`agent_03`) can be recorded but are not used here |
| `agent_01_perc` | % of the polygon's area affected by the primary agent |
| `start_class_01` / `end_class_01` | Vegetation type before/after disturbance (forest_closed, forest_open, shrub, water, etc.) |
| `confidence` | Interpreter's confidence in the disturbance call: 1 (low) - 3 (high) |
| `HUC_12` | Watershed (12-digit hydrologic unit) location of the polygon, populated for 100% of confirmed rows |
| `loc_01` / `loc_02` | Location relative to the park boundary, and country (us/canada) -- the basis of the ISRO/VOYA Canada-correlation finding |
| `owner_type1` | Land ownership category, from the national GAP database |
| `cert_status` / `cert_person` / `cert_date` | Certification tracking |


### A.4 Overall native attribution distribution (agent_01, all parks combined)

In [ ]:
# Native agent-class distribution (change_class = agent_01, NOT collapsed to the SLBE
# report's 6 presentation groups -- native 9-value vocabulary preserved as-is)
agent_counts = glkn["change_class"].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
agent_counts.plot(kind="barh", ax=ax, color="#4575b4")
ax.set_xlabel("confirmed polygons (primary agent = agent_01)")
ax.set_title("GLKN native agent-class distribution (unharmonized)")
ax.invert_yaxis()
fig.tight_layout()
agent_counts


### A.5 Overall temporal distribution (all parks combined)

In [ ]:
by_year = pd.read_csv(QA_DIR / "glkn_by_park_year.csv")
year_area = by_year.groupby("year")["area_m2"].sum() / 1e4

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(year_area.index, year_area.values, color="#4575b4")
ax.set_ylabel("confirmed disturbance area (ha)")
ax.set_xlabel("year")
ax.set_title(f"GLKN confirmed disturbance area by year, all parks combined "
             f"({int(by_year.year.min())}-{int(by_year.year.max())})")
fig.tight_layout()


### A.6 Overall spatial distribution (full dataset)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, park in zip(axes.flat, GLKN_PARKS):
    sub = glkn[glkn["park_code"] == park]
    sub.plot(ax=ax, color="#4575b4", edgecolor="none", alpha=0.6)
    ax.set_title(f"{park} (n={len(sub):,})", fontsize=9)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
axes.flat[-1].axis("off")
fig.suptitle("GLKN confirmed disturbance polygons, per park (full dataset, no boundary overlay)")
fig.tight_layout()


---
## PART B --- 30 m Reference-Grid Assessment

Having characterized GLKN as a whole (Part A), we now assess its **7 analysis subregions** (each park's own `park_code` identity) using the **rasterized 30 m reference-grid pixel summaries** (`src/rasterize_glkn.py`, `src/build_pixel_summary_tables.py`) as the authoritative quantitative source -- not polygon counts or geometric area. Primary (`agent_01`) and all-attributed-agent (`agent_01`+`agent_02`+`agent_03`) views are both preserved as separate parallel products; the figures below use the **primary** view, with all-agent results summarized separately in B.8.

### B.1 Analysis subregions

Each park (`park_code`: APIS, INDU, ISRO, MISS, SACN, SLBE, VOYA) is used directly as its own subregion identity for this quantitative assessment -- **Task 1 retains each park's complete published attributed-polygon geometry and does not clip pixels to any boundary**. See **A.6** above for each subregion's spatial footprint.

**An authoritative study-area AOI now exists for GLKN**: `GLKN_LandTrendr_AOIs`, received 2026-09-22 and confirmed by its own embedded FGDC/Esri lineage to be the actual per-park LandTrendr change-detection analysis extent -- the boundary that generated this reference dataset in the first place, not a boundary being tested after the fact. Full investigation: **Appendix B.6**.

In [ ]:
aoi = pd.read_csv(QA_DIR / "nccn_glkn_aoi_total_pixel_counts.csv")
aoi = aoi[aoi["source"] == "GLKN"][["subregion", "aoi_pixel_count", "aoi_area_ha"]]

pixel_comp = pd.read_csv(QA_DIR / "nccn_glkn_aoi_constraint_comparison_subregion.csv")
pixel_comp = pixel_comp[pixel_comp["source"] == "GLKN-primary"]

table = aoi.merge(pixel_comp[["subregion", "existing_attributed_pixel_years", "pct_diff"]], on="subregion")
table["pct_attributed_pixel_years_in_aoi"] = (100 + table["pct_diff"].fillna(0)).round(3)
table = table.drop(columns=["pct_diff"]).set_index("subregion").reindex(GLKN_PARKS)
table


GLKN's `park_code` assignment and this authoritative AOI are in **essentially complete agreement**: 99.99-100% of every park's existing attributed pixel-years already fall inside its own LandTrendr AOI (only ISRO has any measurable difference: 173 of 1,290,342 pixel-years, 0.01%). Unlike NCCN, there is no meaningful boundary-generation question here. **Task 1's pixel counts are not constrained to this AOI** -- there is essentially nothing to constrain -- consistent with treating GLKN's complete published record as authoritative. As with NCCN, unlabeled pixels within the AOI are not treated as no-change and do not enter the spatial-prevalence calculation below (B.4-B.6). Full investigation: **Appendix B.6**.

### B.2 30 m reference-grid methodology (brief)

Each native `agent_01` class is rasterized independently per subregion x year onto a 30 m grid in the source's native CRS (ESRI:102039), pixel-center inclusion (`all_touched=False`), pixel edges anchored to exact 30 m multiples of the CRS's own (0,0) origin. A pixel may be positive for more than one class in the same year. **Project-defined "30 m reference-grid pixels," not native Landsat/HLS/Prithvi pixels -- alignment must be revisited before model-training sample generation.** Full detail: `outputs/qa/glkn_rasterize_metadata.json`.

### B.3 Total attributed pixel-years and derived hectares (primary)

In [ ]:
ml = pd.read_csv(QA_DIR / "glkn_multilabel_qa_primary.csv")
totals = ml.groupby("subregion")["attributed_pixels"].sum().rename("attributed_pixel_years").reset_index()
totals["derived_area_ha"] = (totals["attributed_pixel_years"] * 0.09).round(1)
totals = totals.set_index("subregion").reindex(GLKN_PARKS).reset_index().sort_values("attributed_pixel_years", ascending=False)
totals


*"Attributed pixel-years" sums each subregion-year's unique attributed-pixel count across years -- a physical pixel attributed in more than one assessment cycle contributes once per year, not once overall.*

### B.4 Native-class spatial prevalence by subregion (primary)

In [ ]:
summary = pd.read_csv(QA_DIR / "glkn_primary_subregion_class_summary.csv")
pivot = summary.pivot(index="subregion", columns="native_class", values="spatial_prevalence_pct").reindex(GLKN_PARKS).fillna(0)
col_order = pivot.sum(axis=0).sort_values(ascending=False).index
pivot = pivot[col_order]

fig, ax = plt.subplots(figsize=(max(8, 0.6 * len(col_order)), 3.4))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd", vmin=0)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=55, ha="right", fontsize=8)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
vmax = pivot.values.max()
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if v > 0:
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7,
                    color="white" if v > vmax * 0.5 else "black")
fig.colorbar(im, ax=ax, label="spatial prevalence (%)")
ax.set_title("GLKN native agent_01 spatial prevalence by subregion (all years, primary)")
fig.tight_layout()


harvest is the single most spatially prevalent agent in 5 of 7 parks (APIS 49%, ISRO 49%, SACN 75%, SLBE 60%, VOYA 87%); development dominates the two smallest/most-developed parks instead (INDU 77%, MISS 99% -- MISS in particular is almost entirely development-attributed). insect_disease_defo is a substantial secondary contributor specifically in APIS and ISRO, absent elsewhere.

### B.5 Overall native-class spatial prevalence (primary)

In [ ]:
ml_total = pd.read_csv(QA_DIR / "glkn_multilabel_qa_primary.csv")["attributed_pixels"].sum()
summary = pd.read_csv(QA_DIR / "glkn_primary_subregion_class_summary.csv")
overall = summary.groupby("native_class")["pixel_count"].sum().sort_values(ascending=False)
overall_pct = (100 * overall / ml_total).round(2)

fig, ax = plt.subplots(figsize=(8, 4))
overall_pct.plot(kind="barh", ax=ax, color="#4575b4")
ax.invert_yaxis()
ax.set_xlabel("spatial prevalence (%), all 7 parks combined")
ax.set_title("GLKN overall native agent_01 spatial prevalence (primary)")
fig.tight_layout()


harvest (56.7%) and insect_disease_defo (28.2%) together account for the large majority of GLKN's attributed pixels dataset-wide; the remaining 8 agent classes are each under 12%, and 5 of them are under 1%.

### B.6 Native-class spatial prevalence through time (primary)

In [ ]:
yc = pd.read_csv(QA_DIR / "glkn_primary_subregion_year_class_summary.csv")

TOP_K_PER_REGION = 3
totals_per_sub_class = yc.groupby(["subregion", "native_class"])["pixel_count"].sum().reset_index()
top_classes = sorted(
    totals_per_sub_class.sort_values("pixel_count", ascending=False)
    .groupby("subregion").head(TOP_K_PER_REGION)["native_class"].unique()
)
yc["class_bucket"] = yc["native_class"].where(yc["native_class"].isin(top_classes), "Other")

palette = plt.get_cmap("tab10").colors
color_map = {cls: palette[i % len(palette)] for i, cls in enumerate(top_classes)}
color_map["Other"] = "#999999"

fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=False)
for ax, park in zip(axes.flat, GLKN_PARKS):
    sub = yc[yc["subregion"] == park]
    pivot = sub.groupby(["year", "class_bucket"])["spatial_prevalence_pct"].sum().unstack(fill_value=0)
    bottom = None
    for cls in list(top_classes) + ["Other"]:
        if cls not in pivot.columns:
            continue
        vals = pivot[cls].values
        ax.bar(pivot.index, vals, bottom=bottom, color=color_map[cls], width=1.0)
        bottom = vals if bottom is None else bottom + vals
    ax.set_title(park, fontsize=9)
    ax.tick_params(labelsize=6)
axes.flat[-1].axis("off")

handles = [plt.Rectangle((0, 0), 1, 1, color=color_map[c]) for c in list(top_classes) + ["Other"]]
fig.legend(handles, list(top_classes) + ["Other"], loc="lower center", ncol=4, fontsize=7, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("GLKN: native agent_01 spatial prevalence (%) by year, within each park\n(each park's own top 3 classes, unioned, + Other; stacked bar height is a magnitude view, not a composition metric)", y=1.03)
fig.tight_layout()


harvest and insect_disease_defo alternate in relative prevalence year to year within APIS and ISRO rather than one permanently dominating; development's high prevalence in INDU/MISS is comparatively steady across their shorter records.

### B.7 Part B limitations (primary)

- **Multi-label pixels**: 0.00% for the primary (agent_01-only) view -- GLKN's near-zero same-year vector overlap (0.005% of area) means essentially no pixel carries two *different* agent_01-attributed polygons in the same year.
- **Small-polygon dropout**: 0 of GLKN's polygons fell under the small-polygon check threshold -- no measurable dropout from pixel-center inclusion.
- **Reference-grid caveat**: see B.2.

### B.8 All-attributed-agent comparison (agent_01 + agent_02 + agent_03)

In [ ]:
primary_total = pd.read_csv(QA_DIR / "glkn_multilabel_qa_primary.csv")["attributed_pixels"].sum()
allagents_total = pd.read_csv(QA_DIR / "glkn_multilabel_qa_allagents.csv")["attributed_pixels"].sum()
primary_ml = pd.read_csv(QA_DIR / "glkn_multilabel_qa_primary.csv")["multi_label_pixels"].sum()
allagents_ml = pd.read_csv(QA_DIR / "glkn_multilabel_qa_allagents.csv")["multi_label_pixels"].sum()

comparison = pd.DataFrame([
    {"view": "primary (agent_01 only)", "attributed_pixel_years": primary_total, "multi_label_pixels": primary_ml,
     "pct_multi_label": round(100 * primary_ml / primary_total, 3)},
    {"view": "all-attributed-agents (01+02+03)", "attributed_pixel_years": allagents_total, "multi_label_pixels": allagents_ml,
     "pct_multi_label": round(100 * allagents_ml / allagents_total, 3)},
])
comparison


The attributed footprint (union of pixels) is identical between the two views (4,077,926 pixel-years) -- agent_02/03 add class labels at already-attributed locations rather than new locations. All-agents raises the multi-label rate from 0.00% to 1.66% (67,840 pixels), confirming agent_02/03 are preserved as genuine additional attributions rather than being dropped or silently merged into agent_01. Per-park all-agent breakdowns: `outputs/qa/glkn_allagents_subregion_class_summary.csv` / `glkn_allagents_subregion_year_class_summary.csv`.

**Note:** the remaining sections of this notebook (Geometry QA, HUC Geography QA, Interactive Maps, Key QA Questions, and the detailed Interpretation/Open Questions writeup including the Isle Royale/Voyageurs Canada investigation) contain detailed processing QA and boundary-definition investigation. This material has been moved to **Appendix B** at the end of this notebook.

---
# ADS Region 6

## PART A --- Source Dataset Description / Characterization

### A.1 What is this dataset, who produced it, and how?

**Source:** the national **Insect & Disease Survey (IDS) database**, maintained by the USDA Forest Service's Forest Health Assessment and Applied Sciences Team (FHAAST) as part of the Forest Health Protection (FHP) program. Authorized by the Cooperative Forestry Assistance Act of 1978, Section 8 [16 U.S.C. 2104], directing the Forest Service to "conduct surveys to detect and appraise insect infestations and disease conditions and man-made stresses affecting trees... and report annually." (USFS *GIS Handbook and Data Conformity Standards*, October 2025, `docs/source_docs/ads/GIS-Handbook-for-Forest-Health-Detection-Survey.pdf`.)

**How it's produced:** the primary collection method is the **aerial detection survey** -- trained observers sketch-map tree damage in real time from an aircraft, using the Digital Mobile Sketch Mapping (DMSM) tablet system (production use since 2016). Ground survey and a developing remote-sensing component also contribute. Surveyors record each observation as a point, polygon, or grid cell, chosen based on how clearly the damage boundary can be discerned from the air -- the Handbook documents this explicitly as a source of surveyor-to-surveyor variability (a "lumper" vs. "splitter" mapping style can produce very different mapped footprints for the same underlying damage). Data pass through post-survey QA/QC before being finalized and compiled into the national IDS database.

**What this dataset explicitly is not:** per the source agency, "Detection surveys do not provide a full inventory of tree damage, but rather are an efficient and economical method of collecting and reporting out on the presence, extent, and severity of forest disturbances." It is a **repeated annual survey**, not a one-time census -- the same ground can be legitimately re-attributed with damage in different years.

**This section's data:** USFS **Region 6 (Pacific Northwest, primarily OR/WA)**, from `ADS_R6_Damage_allyears.shp`, a shapefile export of the same national IDS database (field names truncated to 10 characters, a shapefile fingerprint). Unlike the R10 File Geodatabase (next section), no separate damage-points or surveyed-extent layer was provided for R6 -- only this polygon export.

**Years:** the national IDS archive extends back to **1997**; this extract spans **1997-2025 (29 distinct years)**.

**What one record represents:** a single surveyor observation of tree damage at a location in a given survey year -- one specific host / damage-causal-agent / damage-type combination, at the surveyor's chosen feature type and drawn footprint. It is **not** a 1:1 stand-in for a discrete real-world disturbance event, and it is **not evidence of unique land area disturbed when summed across years** -- cumulative record/area totals are a repeated-survey sum, not a one-time census. Multiple observations legitimately sharing the same or overlapping footprint ("pancaked" features) are expected and documented by the source agency, not duplicates or errors: 0 of this section's 911,911 records are flagged `OBSERVATION_COUNT=='MULTIPLE'`.

In [ ]:
ads_r6 = gpd.read_parquet(DATA_PROCESSED / "ads_r6" / "ads_r6_region6_with_ecoregion.parquet")
ecoregions = gpd.read_parquet(DATA_PROCESSED / "boundaries" / "ads_r6_ecoregions.parquet")

print(f"Loaded {len(ads_r6):,} ADS R6 (REGION_ID==6) damage polygons, CRS={ads_r6.crs.to_epsg()}")
print(f"Loaded {len(ecoregions)} dissolved EPA Level III ecoregions")
print(f"Years represented: {int(ads_r6['SURVEY_YEA'].min())}-{int(ads_r6['SURVEY_YEA'].max())} "
      f"({ads_r6['SURVEY_YEA'].nunique()} distinct years)")


**Region filter QA** (re-verified, not assumed): the raw ADS R6 file contains a small number of stray non-Region-6 records despite the filename.

In [ ]:
region_qa = pd.read_csv(QA_DIR / "ads_r6_region_filter_qa.csv")
region_qa


### A.2 Basic dataset summary (this section, before any regional split)

In [ ]:
summary_row = pd.DataFrame([{
    "record_count": len(ads_r6),
    "attributed_area_ha": ads_r6["area_m2"].sum() / 1e4,
    "first_year": int(ads_r6["SURVEY_YEA"].min()),
    "last_year": int(ads_r6["SURVEY_YEA"].max()),
    "years_represented": int(ads_r6["SURVEY_YEA"].nunique()),
    "n_DCA_classes": int(ads_r6["DCA_CODE"].nunique()),
    "n_DAMAGE_TYPE_classes": int(ads_r6["DAMAGE_T_1"].nunique()),
}])
summary_row


### A.3 Attribute / schema table

Focused on fields relevant to interpreting and using the reference data -- not every GIS field.

| Field | Meaning |
|---|---|
| `SURVEY_YEAR` | Year the survey was conducted |
| `REGION_ID` | USFS Region identifier (6 = Region 6 (Pacific Northwest, primarily OR/WA); 1,254 stray non-R6 records (`REGION_ID` 1 and 5) filtered out -- verified again rather than assumed, see below) |
| `DCA_CODE` / `DCA_COMMON` | **Damage Causal Agent** -- the specific insect, disease, or abiotic agent responsible (national code list, 1,000+ possible codes; this section uses 91 distinct codes) |
| `DAMAGE_TYP` / `DAMAGE_T_1` | Type of damage observed (e.g., mortality, discoloration, defoliation) -- a second, coarser native taxonomy dimension alongside DCA |
| `PERCENT_AFFECTED_CODE` / `PERCENT_AFFECTED` / `PERCENT_MIN`/`MAX`/`MID` | **Damage intensity**: % of standing (live + dead) trees *within the polygon* affected, on a 5-class scale (Very Light 1-3% to Very Severe >50%) -- not the % of the polygon's total area |
| `HOST` / `HOST_GROUP` | Tree species or host group affected |
| `OBSERVATION_COUNT` / `OBSERVAT_1` | Flags whether a footprint has multiple co-located ("pancaked") observations |
| `AREA_TYPE` | Feature type recorded (POLYGON here) |
| `COLLECTION_MODE` | Aerial_Survey / Ground_Check / Ground_Survey / Scan_Sketch / DesktopGIS -- how the observation was collected |
| `OBJECTID` | Row/footprint identifier |
| `STATUS` | Data-finality flag in this extract |


**Label geometry note:** ADS damage footprints are drawn by surveyors sketch-mapping observed damage from the air (or occasionally on the ground) onto a base map, not delineated by a fixed, objective boundary-detection algorithm. The USFS GIS Handbook documents that the same underlying damage can be mapped as a tighter "splitter" outline or a looser "lumper" outline depending on surveyor judgment, and can be recorded as a point, polygon, or grid cell depending on how discernible the boundary is from the air. Mapped polygon boundaries should therefore be treated as a survey-derived approximation of where damage occurred, not an exact ground-truthed boundary, when later used as training or evaluation labels.

### A.4 Overall native attribution distribution (DCA and DAMAGE_TYPE, all regions combined)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

dca_area = ads_r6.groupby("DCA_COMMON")["area_m2"].sum().sort_values(ascending=False).head(15) / 1e4
ax1.barh(dca_area.index[::-1], dca_area.values[::-1], color="#4575b4")
ax1.set_xlabel("attributed area (ha)")
ax1.set_title(f"ADS R6 top 15 DCA (of {ads_r6['DCA_CODE'].nunique()} total), all regions combined")

dmg_area = ads_r6.groupby("DAMAGE_T_1")["area_m2"].sum().sort_values(ascending=False) / 1e4
ax2.barh(dmg_area.index[::-1], dmg_area.values[::-1], color="#d73027")
ax2.set_xlabel("attributed area (ha)")
ax2.set_title(f"ADS R6 DAMAGE_T_1 ({ads_r6['DAMAGE_T_1'].nunique()} total), all regions combined")

fig.tight_layout()


The causal-agent distribution is strongly uneven: mountain pine beetle alone accounts for roughly 4.33 of the 20.3 million ha total (about 21%), followed by fir engraver (~3.46 million ha); the great majority of the 91 distinct DCA codes each contribute a small fraction of that. By damage type, Mortality dominates overwhelmingly (~13.58 million ha, roughly 67% of total area), with Defoliation a distant second (~3.73 million ha). **For reference-data use:** abundant classes like mountain pine beetle and Mortality would supply far more candidate training/evaluation examples than most DCA codes, which appear in comparatively few records -- a consideration for later sampling and focal-domain design, not resolved here.

### A.5 Overall spatial distribution (full dataset)

A 2D density view of ADS R6 polygon centroids handles all ~912,000 points natively (unlike plotting every polygon outline) and directly answers "where are the concentrations" -- the actual question motivating this analysis.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ecoregions.boundary.plot(ax=ax, color="black", linewidth=1)
centroids = ads_r6.geometry.centroid
hb = ax.hexbin(centroids.x, centroids.y, gridsize=80, cmap="inferno", bins="log", mincnt=1)
fig.colorbar(hb, ax=ax, label="log10(polygon count) per hex cell")
ax.set_title(f"ADS R6 damage polygon density (n={len(ads_r6):,}), EPA ecoregion boundaries for context")
ax.set_aspect("equal")
fig.tight_layout()


Attributed area is concentrated in a relatively small number of ecoregions rather than spread evenly: Eastern Cascades Slopes and Foothills alone accounts for roughly 21% of total attributed area (4.19 of 20.3 million ha, B.2), and the top 3 ecoregions together account for more than half the total, while Klamath Mountains/California High North Coast Range contributes only about 4%.

### A.6 Overall temporal distribution (all regions combined)

In [ ]:
by_year = pd.read_csv(QA_DIR / "ads_r6_by_ecoregion_year.csv")
year_area = by_year.groupby("SURVEY_YEA")["area_m2"].sum() / 1e4  # ha, all ecoregions combined

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(year_area.index, year_area.values, color="#4575b4")
ax.set_ylabel("attributed area (ha)")
ax.set_xlabel("survey year")
ax.set_title(f"ADS R6 attributed area by year, all ecoregions combined "
             f"({int(by_year.SURVEY_YEA.min())}-{int(by_year.SURVEY_YEA.max())}, "
             f"{by_year.SURVEY_YEA.nunique()} distinct years)")
fig.tight_layout()


Attributed area shows moderate, fairly continuous year-to-year variation for most of the record (roughly 0.3-1.1 million ha per year, 1997-2020), but a marked recent intensification stands out: 2021-2023 are the three highest years on record, peaking at roughly 1.54 million ha in 2022 -- noticeably above the historical baseline.

*(An illustrative random-polygon sample for shape inspection is provided in Appendix C rather than the main narrative.)*

---
## PART B --- 30 m Reference-Grid Assessment

Having characterized ADS R6 as a whole (Part A), we now assess its **7 analysis subregions** (dissolved EPA Level III ecoregions) using the **rasterized 30 m reference-grid pixel summaries** (`src/rasterize_ads_r6.py`, `src/build_pixel_summary_tables.py`) as the authoritative quantitative source -- not polygon counts or geometric area. Subregion membership uses actual ecoregion boundary geometry (pixel-center-in-subregion), not the earlier centroid-based vector-stage assignment. **DCA is the primary attribution dimension** here; Damage Type is a separate, secondary product (B.8).

### B.1 Analysis subregions: EPA Level III ecoregions (dissolved)

In [ ]:
ecoregions[["us_l3code", "us_l3name", "n_parts", "region_area_m2"]].assign(
    region_area_km2=lambda d: d["region_area_m2"] / 1e6
)[["us_l3code", "us_l3name", "n_parts", "region_area_km2"]]


In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ecoregions.plot(ax=ax, column="us_l3name", cmap="tab10", alpha=0.5, edgecolor="black", linewidth=0.8, legend=True,
                 legend_kwds={"loc": "lower left", "fontsize": 7})
ax.set_title("BugNet R6 EPA Level III ecoregions, dissolved (7 regions)")
ax.set_aspect("equal")
fig.tight_layout()


### B.2 30 m reference-grid methodology (brief)

Each native `DCA_CODE` class is rasterized independently per ecoregion x year onto a 30 m grid in the source's native CRS (ESRI:102039), pixel-center inclusion (`all_touched=False`), pixel edges anchored to exact 30 m multiples of the CRS's own (0,0) origin. A pixel may be positive for more than one DCA in the same year -- consistent with the same-year overlap investigation (97% of R6 same-year overlap is the documented "pancake" pattern of multiple legitimate attributions at one footprint, not a data problem). **Project-defined "30 m reference-grid pixels," not native Landsat/HLS/Prithvi pixels -- alignment must be revisited before model-training sample generation.** Full detail: `outputs/qa/ads_r6_rasterize_metadata.json`.

### B.3 Total attributed pixel-years and derived hectares (DCA, primary)

In [ ]:
ml = pd.read_csv(QA_DIR / "ads_r6_subregion_year_dca_multilabel_qa.csv")
totals = ml.groupby(["subregion", "subregion_name"])["attributed_pixels"].sum().rename("attributed_pixel_years").reset_index()
totals["derived_area_ha"] = (totals["attributed_pixel_years"] * 0.09).round(1)
totals = totals.sort_values("attributed_pixel_years", ascending=False)
totals


*"Attributed pixel-years" sums each ecoregion-year's unique attributed-pixel count across years -- ADS is a repeated annual survey, so a physical pixel legitimately attributed in multiple different years contributes once per year, not once overall.*

### B.4 Native DCA spatial prevalence by subregion

In [ ]:
summary = pd.read_csv(QA_DIR / "ads_r6_dca_subregion_class_summary.csv")
TOP_N_COLS = 20  # cap columns for legibility; DCA has 91 codes total
top_cols = summary.groupby("native_class")["pixel_count"].sum().sort_values(ascending=False).head(TOP_N_COLS).index
pivot = summary[summary["native_class"].isin(top_cols)].pivot(index="subregion_name", columns="native_class", values="spatial_prevalence_pct").fillna(0)
pivot = pivot[top_cols]
region_order = summary.groupby("subregion_name")["pixel_count"].sum().sort_values(ascending=False).index
pivot = pivot.reindex(region_order)

fig, ax = plt.subplots(figsize=(max(10, 0.5 * len(top_cols)), 4))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd", vmin=0)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=60, ha="right", fontsize=7)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=8)
vmax = pivot.values.max()
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if v > 5:
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=6,
                    color="white" if v > vmax * 0.5 else "black")
fig.colorbar(im, ax=ax, label="spatial prevalence (%)")
ax.set_title(f"ADS R6 native DCA spatial prevalence by ecoregion (all years, top {TOP_N_COLS} of 91 DCA codes)")
fig.tight_layout()


Each ecoregion is dominated by a different agent: Swiss needle cast in Coast Range (45%), mountain pine beetle in Eastern Cascades Slopes and Foothills (41%) and Cascades (24%), fir engraver in Blue Mountains (37%), western spruce budworm in North Cascades (37%), and flatheaded fir borer in Klamath Mountains (50%) -- Coast Range and Klamath Mountains are each dominated by an agent that ranks outside the dataset-wide top eight DCA codes (B.5), a genuinely region-specific signal that a dataset-wide ranking alone would hide.

### B.5 Overall native DCA spatial prevalence

In [ ]:
ml_total = pd.read_csv(QA_DIR / "ads_r6_subregion_year_dca_multilabel_qa.csv")["attributed_pixels"].sum()
summary = pd.read_csv(QA_DIR / "ads_r6_dca_subregion_class_summary.csv")
overall = summary.groupby("native_class")["pixel_count"].sum().sort_values(ascending=False).head(15)
overall_pct = (100 * overall / ml_total).round(2)

fig, ax = plt.subplots(figsize=(8, 5))
overall_pct.plot(kind="barh", ax=ax, color="#4575b4")
ax.invert_yaxis()
ax.set_xlabel("spatial prevalence (%), all 7 ecoregions combined")
ax.set_title("ADS R6 top 15 DCA overall spatial prevalence (of 91 total codes)")
fig.tight_layout()


mountain pine beetle (22.6%), fir engraver (19.3%), and western spruce budworm (15.4%) together account for over half of all attributed pixel-years dataset-wide; the remaining 88 DCA codes are each under 10%.

### B.6 Native DCA spatial prevalence through time

In [ ]:
yc = pd.read_csv(QA_DIR / "ads_r6_dca_subregion_year_class_summary.csv")

TOP_K_PER_REGION = 3
totals_per_sub_class = yc.groupby(["subregion_name", "native_class"])["pixel_count"].sum().reset_index()
top_classes = sorted(
    totals_per_sub_class.sort_values("pixel_count", ascending=False)
    .groupby("subregion_name").head(TOP_K_PER_REGION)["native_class"].unique()
)
yc["class_bucket"] = yc["native_class"].where(yc["native_class"].isin(top_classes), "Other")

palette = plt.get_cmap("tab20").colors
color_map = {cls: palette[i % len(palette)] for i, cls in enumerate(top_classes)}
color_map["Other"] = "#999999"

region_order = summary.groupby("subregion_name")["pixel_count"].sum().sort_values(ascending=False).index.tolist()
ncols = 4
nrows = -(-len(region_order) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 2.8 * nrows), sharex=False)
axes_flat = axes.flat
for ax, region in zip(axes_flat, region_order):
    sub = yc[yc["subregion_name"] == region]
    pivot = sub.groupby(["year", "class_bucket"])["spatial_prevalence_pct"].sum().unstack(fill_value=0)
    bottom = None
    for cls in list(top_classes) + ["Other"]:
        if cls not in pivot.columns:
            continue
        vals = pivot[cls].values
        ax.bar(pivot.index, vals, bottom=bottom, color=color_map[cls], width=1.0)
        bottom = vals if bottom is None else bottom + vals
    ax.set_title(region, fontsize=8)
    ax.tick_params(labelsize=6)
for ax in list(axes_flat)[len(region_order):]:
    ax.axis("off")

handles = [plt.Rectangle((0, 0), 1, 1, color=color_map[c]) for c in list(top_classes) + ["Other"]]
fig.legend(handles, list(top_classes) + ["Other"], loc="lower center", ncol=4, fontsize=6, bbox_to_anchor=(0.5, -0.05))
fig.suptitle("ADS R6: DCA spatial prevalence (%) by year, within each ecoregion\n(each region's own top 3 DCA, unioned, + Other; stacked bar height is a magnitude view, not a composition metric)", y=1.03)
fig.tight_layout()


Consistent with the earlier region x year finding, prevalence within each ecoregion is concentrated in specific multi-year runs rather than a steady background rate -- e.g. mountain pine beetle's prevalence in Eastern Cascades Slopes and Foothills and Cascades rises sharply in the most recent (2022-2024) years relative to its longer-run baseline.

### B.7 Part B limitations (DCA)

- **Multi-label pixels**: 8.21% of R6's attributed pixels carry more than one DCA in the same year -- the highest multi-label rate of the four sources, driven almost entirely (97% of same-year overlap) by the documented "pancake" pattern (multiple legitimate co-located observations), not a data-quality problem. See A.1/B.2.
- **Small-polygon dropout**: 1,373 of 2,342 checked small polygons (58.6%) contributed zero pixels under center-inclusion -- the highest dropout rate of the four sources, reflecting ADS's legitimately small point/grid-cell-derived observations. Only 12 of these were outside all analysis subregions entirely (irrelevant to the summary); the rest are real small-polygon dropout. Full list: `outputs/qa/ads_r6_zero_pixel_polygons.csv`.
- **Reference-grid caveat**: see B.2.

### B.8 Damage Type (secondary)

In [ ]:
dmg_summary = pd.read_csv(QA_DIR / "ads_r6_damagetype_subregion_class_summary.csv")
ml_dmg_total = pd.read_csv(QA_DIR / "ads_r6_subregion_year_damagetype_multilabel_qa.csv")["attributed_pixels"].sum()
overall_dmg = dmg_summary.groupby("native_class")["pixel_count"].sum().sort_values(ascending=False)
overall_dmg_pct = (100 * overall_dmg / ml_dmg_total).round(2)

fig, ax = plt.subplots(figsize=(8, 4))
overall_dmg_pct.plot(kind="barh", ax=ax, color="#d73027")
ax.invert_yaxis()
ax.set_xlabel("spatial prevalence (%), all 7 ecoregions combined")
ax.set_title("ADS R6 overall DAMAGE_TYP spatial prevalence (secondary taxonomy)")
fig.tight_layout()


Mortality is overwhelmingly the most spatially prevalent damage type dataset-wide (matching the vector-area-based finding in A.4), a coarser and less locally-differentiated signal than DCA -- damage type does not show the same ecoregion-specific dominance pattern seen in DCA (B.4). Full per-ecoregion breakdown: `outputs/qa/ads_r6_damagetype_subregion_class_summary.csv`.

**Note:** the illustrative polygon sample (A.7-equivalent, see the pointer above) and processing-history detail (naive-overlay performance fix, the per-ecoregion area-crediting bug fix) have been moved to **Appendix C** at the end of this notebook.

## Interpretation / Summary

### Not yet done (by design, per instruction)

- No cross-source class harmonization (native `DCA_CODE`/`DAMAGE_TYP` preserved as-is).
- No attempt to reconstruct ADS's original historical analysis boundaries.
- No further optimization of the 97.0% ecoregion capture rate.
- No final reference-data summary statistics, no focal-area selection.

---
# ADS Region 10

## PART A --- Source Dataset Description / Characterization

### A.1 What is this dataset, who produced it, and how?

**Source:** the national **Insect & Disease Survey (IDS) database**, maintained by the USDA Forest Service's Forest Health Assessment and Applied Sciences Team (FHAAST) as part of the Forest Health Protection (FHP) program. Authorized by the Cooperative Forestry Assistance Act of 1978, Section 8 [16 U.S.C. 2104], directing the Forest Service to "conduct surveys to detect and appraise insect infestations and disease conditions and man-made stresses affecting trees... and report annually." (USFS *GIS Handbook and Data Conformity Standards*, October 2025, `docs/source_docs/ads/GIS-Handbook-for-Forest-Health-Detection-Survey.pdf`.)

**How it's produced:** the primary collection method is the **aerial detection survey** -- trained observers sketch-map tree damage in real time from an aircraft, using the Digital Mobile Sketch Mapping (DMSM) tablet system (production use since 2016). Ground survey and a developing remote-sensing component also contribute. Surveyors record each observation as a point, polygon, or grid cell, chosen based on how clearly the damage boundary can be discerned from the air -- the Handbook documents this explicitly as a source of surveyor-to-surveyor variability (a "lumper" vs. "splitter" mapping style can produce very different mapped footprints for the same underlying damage). Data pass through post-survey QA/QC before being finalized and compiled into the national IDS database.

**What this dataset explicitly is not:** per the source agency, "Detection surveys do not provide a full inventory of tree damage, but rather are an efficient and economical method of collecting and reporting out on the presence, extent, and severity of forest disturbances." It is a **repeated annual survey**, not a one-time census -- the same ground can be legitimately re-attributed with damage in different years.

**This section's data:** USFS **Region 10 (Alaska)**, from `DAMAGE_AREAS_FLAT_AllYears_AK_Rgn10`, a File Geodatabase layer (`AK_Region10_AllYears.gdb`, extracted from the raw ZIP to a persistent working copy; raw ZIP itself untouched). Field names are untruncated (unlike the ADS R6 shapefile export in the previous section), and this GDB also contains a damage-points layer and a dedicated surveyed-area-extent layer (not yet used in this assessment).

**Years:** the national IDS archive extends back to **1997**; this extract spans **1997-2025 (29 distinct years)**.

**What one record represents:** a single surveyor observation of tree damage at a location in a given survey year -- one specific host / damage-causal-agent / damage-type combination, at the surveyor's chosen feature type and drawn footprint. It is **not** a 1:1 stand-in for a discrete real-world disturbance event, and it is **not evidence of unique land area disturbed when summed across years** -- cumulative record/area totals are a repeated-survey sum, not a one-time census. Multiple observations legitimately sharing the same or overlapping footprint ("pancaked" features) are expected and documented by the source agency, not duplicates or errors: 6,949 of this section's 151,309 records are flagged `OBSERVATION_COUNT=='MULTIPLE'`.

In [ ]:
ads_r10 = gpd.read_parquet(DATA_PROCESSED / "ads_r10" / "ads_r10_with_huc6.parquet")
huc6 = gpd.read_parquet(DATA_PROCESSED / "boundaries" / "ads_r10_huc6.parquet")

print(f"Loaded {len(ads_r10):,} ADS R10 (REGION_ID==10) damage polygons, CRS={ads_r10.crs.to_epsg()}")
print(f"Loaded {len(huc6)} BugNet R10 HUC6 regions")
print(f"Years represented: {int(ads_r10['SURVEY_YEAR'].min())}-{int(ads_r10['SURVEY_YEAR'].max())} "
      f"({ads_r10['SURVEY_YEAR'].nunique()} distinct years)")
print(f"100% REGION_ID==10 -- no region filter needed (unlike R6, which had stray non-6 records).")

n_multi = int((ads_r10["OBSERVATION_COUNT"] == "MULTIPLE").sum())
n_distinct_footprint = ads_r10["DAMAGE_AREA_ID"].nunique()
print(f"\nPancake QA: {n_multi:,} rows flagged OBSERVATION_COUNT=='MULTIPLE'; "
      f"{n_distinct_footprint:,} distinct DAMAGE_AREA_ID footprints of {len(ads_r10):,} total rows "
      f"({len(ads_r10) - n_distinct_footprint:,} 'extra' rows -- legitimate distinct co-located "
      f"observations per the official USDA readme, not duplicates).")


### A.2 Basic dataset summary (this section, before any regional split)

In [ ]:
summary_row = pd.DataFrame([{
    "record_count": len(ads_r10),
    "attributed_area_ha": ads_r10["area_m2"].sum() / 1e4,
    "first_year": int(ads_r10["SURVEY_YEAR"].min()),
    "last_year": int(ads_r10["SURVEY_YEAR"].max()),
    "years_represented": int(ads_r10["SURVEY_YEAR"].nunique()),
    "n_DCA_classes": int(ads_r10["DCA_CODE"].nunique()),
    "n_DAMAGE_TYPE_classes": int(ads_r10["DAMAGE_TYPE"].nunique()),
}])
summary_row


### A.3 Attribute / schema table

Focused on fields relevant to interpreting and using the reference data -- not every GIS field.

| Field | Meaning |
|---|---|
| `SURVEY_YEAR` | Year the survey was conducted |
| `REGION_ID` | USFS Region identifier (10 = Region 10 (Alaska); verified 100% here, no filtering needed) |
| `DCA_CODE` / `DCA_COMMON_NAME` | **Damage Causal Agent** -- the specific insect, disease, or abiotic agent responsible (national code list, 1,000+ possible codes; this section uses 70 distinct codes) |
| `DAMAGE_TYPE_CODE` / `DAMAGE_TYPE` | Type of damage observed (e.g., mortality, discoloration, defoliation) -- a second, coarser native taxonomy dimension alongside DCA |
| `PERCENT_AFFECTED_CODE` / `PERCENT_AFFECTED` / `PERCENT_MIN`/`MAX`/`MID` | **Damage intensity**: % of standing (live + dead) trees *within the polygon* affected, on a 5-class scale (Very Light 1-3% to Very Severe >50%) -- not the % of the polygon's total area |
| `HOST` / `HOST_GROUP` | Tree species or host group affected |
| `OBSERVATION_COUNT` / `OBSERVATION_ID` | Flags whether a footprint has multiple co-located ("pancaked") observations |
| `AREA_TYPE` | Feature type recorded (POLYGON here) |
| `COLLECTION_MODE` | Aerial_Survey / Ground_Check / Ground_Survey / Scan_Sketch / DesktopGIS -- how the observation was collected |
| `DAMAGE_AREA_ID` | Row/footprint identifier |
| `STATUS` | Data-finality flag in this extract |


**Label geometry note:** ADS damage footprints are drawn by surveyors sketch-mapping observed damage from the air (or occasionally on the ground) onto a base map, not delineated by a fixed, objective boundary-detection algorithm. The USFS GIS Handbook documents that the same underlying damage can be mapped as a tighter "splitter" outline or a looser "lumper" outline depending on surveyor judgment, and can be recorded as a point, polygon, or grid cell depending on how discernible the boundary is from the air. Mapped polygon boundaries should therefore be treated as a survey-derived approximation of where damage occurred, not an exact ground-truthed boundary, when later used as training or evaluation labels.

### A.4 Overall native attribution distribution (DCA and DAMAGE_TYPE, all regions combined)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

dca_area = ads_r10.groupby("DCA_COMMON_NAME")["area_m2"].sum().sort_values(ascending=False).head(15) / 1e4
ax1.barh(dca_area.index[::-1], dca_area.values[::-1], color="#4575b4")
ax1.set_xlabel("attributed area (ha)")
ax1.set_title(f"ADS R10 top 15 DCA (of {ads_r10['DCA_CODE'].nunique()} total), all regions combined")

dmg_area = ads_r10.groupby("DAMAGE_TYPE")["area_m2"].sum().sort_values(ascending=False) / 1e4
ax2.barh(dmg_area.index[::-1], dmg_area.values[::-1], color="#d73027")
ax2.set_xlabel("attributed area (ha)")
ax2.set_title(f"ADS R10 DAMAGE_TYPE ({ads_r10['DAMAGE_TYPE'].nunique()} total), all regions combined")

fig.tight_layout()


The causal-agent distribution is strongly uneven: aspen leafminer alone accounts for roughly 2.37 of the 9.8 million ha total (about 24%), followed by spruce beetle (~2.12 million ha); most of the 70 distinct DCA codes each contribute a small fraction of that. By damage type, Defoliation dominates (~4.58 million ha, roughly 47% of total area), with Mortality a distant second (~2.33 million ha). **For reference-data use:** abundant classes like aspen leafminer and Defoliation would supply far more candidate training/evaluation examples than most DCA codes, which appear in comparatively few records -- a consideration for later sampling and focal-domain design, not resolved here.

### A.5 Overall spatial distribution (full dataset)

A 2D density view of ADS R10 polygon centroids handles all ~151,000 points natively -- the actual question motivating this analysis.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
huc6.boundary.plot(ax=ax, color="black", linewidth=1)
centroids = ads_r10.geometry.centroid
hb = ax.hexbin(centroids.x, centroids.y, gridsize=80, cmap="inferno", bins="log", mincnt=1)
fig.colorbar(hb, ax=ax, label="log10(polygon count) per hex cell")
ax.set_title(f"ADS R10 damage polygon density (n={len(ads_r10):,}), HUC6 boundaries for context")
ax.set_aspect("equal")
fig.tight_layout()


Attributed area is concentrated in a relatively small number of HUC6 basins rather than spread evenly: Tanana River alone accounts for roughly 20% of total attributed area (1.96 of 9.8 million ha, B.2), and the top 5 basins together account for more than half the total, while the smallest basins (e.g., Kobuk-Selawik Rivers) each contribute under half a percent.

### A.6 Overall temporal distribution (all regions combined)

In [ ]:
by_year = pd.read_csv(QA_DIR / "ads_r10_by_huc6_year.csv")
year_area = by_year.groupby("SURVEY_YEAR")["area_m2"].sum() / 1e4  # ha, all HUC6 combined

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(year_area.index, year_area.values, color="#4575b4")
ax.set_ylabel("attributed area (ha)")
ax.set_xlabel("survey year")
ax.set_title(f"ADS R10 attributed area by year, all HUC6 regions combined "
             f"({int(by_year.SURVEY_YEAR.min())}-{int(by_year.SURVEY_YEAR.max())}, "
             f"{by_year.SURVEY_YEAR.nunique()} distinct years)")
fig.tight_layout()


Unlike NCCN and GLKN, ADS R10's attributed area is fairly continuous across its 29-year span rather than concentrated in one or two outlier years -- annual totals range roughly between 100,000 and 650,000 ha, consistent with ADS being a repeated annual survey rather than an episodic mapping effort. The single highest year is 2021 (~650,000 ha); no year dominates the record the way 2001 or 2017 do for NCCN/GLKN (see those sections).

*(An illustrative random-polygon sample for shape inspection is provided in Appendix C rather than the main narrative.)*

---
## PART B --- 30 m Reference-Grid Assessment

Having characterized ADS R10 as a whole (Part A), we now assess its **20 analysis subregions** (HUC6 watershed basins) using the **rasterized 30 m reference-grid pixel summaries** (`src/rasterize_ads_r10.py`, `src/build_pixel_summary_tables.py`) as the authoritative quantitative source -- not polygon counts or geometric area. Subregion membership uses actual HUC6 boundary geometry (pixel-center-in-subregion), not the earlier centroid-based vector-stage assignment. **DCA is the primary attribution dimension** here; Damage Type is a separate, secondary product (B.8).

### B.1 Analysis subregions: HUC6 watershed basins (no dissolve needed)

In [ ]:
huc6[["huc6", "name", "states", "n_parts"]].assign(
    region_area_km2=lambda d: huc6["region_area_m2"] / 1e6
)[["huc6", "name", "states", "n_parts", "region_area_km2"]]


In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
huc6.plot(ax=ax, column="name", cmap="tab20", alpha=0.5, edgecolor="black", linewidth=0.8, legend=True,
          legend_kwds={"loc": "lower left", "fontsize": 6, "ncol": 2})
ax.set_title("BugNet R10 HUC6 regions, Alaska (20 regions, no dissolve needed)")
ax.set_aspect("equal")
fig.tight_layout()


### B.2 30 m reference-grid methodology (brief)

Each native `DCA_COMMON_NAME` class is rasterized independently per HUC6 basin x year onto a 30 m grid in the source's native CRS (NAD83 / Alaska Albers, EPSG:3338), pixel-center inclusion (`all_touched=False`), pixel edges anchored to exact 30 m multiples of the CRS's own (0,0) origin. A pixel may be positive for more than one DCA in the same year -- consistent with the same-year overlap investigation for the ADS sources generally (the same "pancake" pattern of multiple legitimate co-located observations documented for R6, not a data problem). **Project-defined "30 m reference-grid pixels," not native Landsat/HLS/Prithvi pixels -- alignment must be revisited before model-training sample generation.** Full detail: `outputs/qa/ads_r10_rasterize_metadata.json`.

### B.3 Total attributed pixel-years and derived hectares (DCA, primary)

In [ ]:
ml = pd.read_csv(QA_DIR / "ads_r10_subregion_year_dca_multilabel_qa.csv")
totals = ml.groupby(["subregion", "subregion_name"])["attributed_pixels"].sum().rename("attributed_pixel_years").reset_index()
totals["derived_area_ha"] = (totals["attributed_pixel_years"] * 0.09).round(1)
totals = totals.sort_values("attributed_pixel_years", ascending=False)
totals


*"Attributed pixel-years" sums each HUC6-year's unique attributed-pixel count across years -- ADS is a repeated annual survey, so a physical pixel legitimately attributed in multiple different years contributes once per year, not once overall.*

### B.4 Native DCA spatial prevalence by subregion

In [ ]:
summary = pd.read_csv(QA_DIR / "ads_r10_dca_subregion_class_summary.csv")
TOP_N_COLS = 20  # cap columns for legibility; DCA has 67 codes total
top_cols = summary.groupby("native_class")["pixel_count"].sum().sort_values(ascending=False).head(TOP_N_COLS).index
pivot = summary[summary["native_class"].isin(top_cols)].pivot(index="subregion_name", columns="native_class", values="spatial_prevalence_pct").fillna(0)
pivot = pivot[top_cols]
region_order = summary.groupby("subregion_name")["pixel_count"].sum().sort_values(ascending=False).index
pivot = pivot.reindex(region_order)

fig, ax = plt.subplots(figsize=(max(10, 0.5 * len(top_cols)), 7))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd", vmin=0)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=60, ha="right", fontsize=7)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=7)
vmax = pivot.values.max()
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if v > 5:
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=6,
                    color="white" if v > vmax * 0.5 else "black")
fig.colorbar(im, ax=ax, label="spatial prevalence (%)")
ax.set_title(f"ADS R10 native DCA spatial prevalence by HUC6 basin (all years, top {TOP_N_COLS} of 67 DCA codes)")
fig.tight_layout()


Several basins are dominated by a single agent well above 50% spatial prevalence: aspen leafminer in Tanana River (68.5%) and Porcupine River (59.3%), spruce beetle in Kenai Peninsula (71.4%) and Susitna River (68.0%), and western blackheaded budworm in Prince William Sound (71.9%). Birch leafroller -- ranked 12th dataset-wide (1.8%, outside the top ten in B.5) -- is nonetheless a top-two agent in Outlet Yukon River (23.2%) and Lower Kuskokwim River (38.2%), a locally dominant but globally minor signal comparable to R6's Coast Range/Klamath Mountains pattern.

### B.5 Overall native DCA spatial prevalence

In [ ]:
ml_total = pd.read_csv(QA_DIR / "ads_r10_subregion_year_dca_multilabel_qa.csv")["attributed_pixels"].sum()
summary = pd.read_csv(QA_DIR / "ads_r10_dca_subregion_class_summary.csv")
overall = summary.groupby("native_class")["pixel_count"].sum().sort_values(ascending=False).head(15)
overall_pct = (100 * overall / ml_total).round(2)

fig, ax = plt.subplots(figsize=(8, 5))
overall_pct.plot(kind="barh", ax=ax, color="#4575b4")
ax.invert_yaxis()
ax.set_xlabel("spatial prevalence (%), all 20 HUC6 basins combined")
ax.set_title("ADS R10 top 15 DCA overall spatial prevalence (of 67 total codes)")
fig.tight_layout()


aspen leafminer (27.5%) and spruce beetle (18.5%) together account for nearly half of all attributed pixel-years dataset-wide; willow leaf blotchminer (9.5%) and western blackheaded budworm (7.3%) are a clear secondary tier, and the remaining 63 DCA codes are each under 6%.

### B.6 Native DCA spatial prevalence through time

In [ ]:
yc = pd.read_csv(QA_DIR / "ads_r10_dca_subregion_year_class_summary.csv")

TOP_K_PER_REGION = 3
totals_per_sub_class = yc.groupby(["subregion_name", "native_class"])["pixel_count"].sum().reset_index()
top_classes = sorted(
    totals_per_sub_class.sort_values("pixel_count", ascending=False)
    .groupby("subregion_name").head(TOP_K_PER_REGION)["native_class"].unique()
)
yc["class_bucket"] = yc["native_class"].where(yc["native_class"].isin(top_classes), "Other")

palette = plt.get_cmap("tab20").colors
color_map = {cls: palette[i % len(palette)] for i, cls in enumerate(top_classes)}
color_map["Other"] = "#999999"

region_order = summary.groupby("subregion_name")["pixel_count"].sum().sort_values(ascending=False).index.tolist()
ncols = 4
nrows = -(-len(region_order) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 2.8 * nrows), sharex=False)
axes_flat = axes.flat
for ax, region in zip(axes_flat, region_order):
    sub = yc[yc["subregion_name"] == region]
    pivot = sub.groupby(["year", "class_bucket"])["spatial_prevalence_pct"].sum().unstack(fill_value=0)
    bottom = None
    for cls in list(top_classes) + ["Other"]:
        if cls not in pivot.columns:
            continue
        vals = pivot[cls].values
        ax.bar(pivot.index, vals, bottom=bottom, color=color_map[cls], width=1.0)
        bottom = vals if bottom is None else bottom + vals
    ax.set_title(region, fontsize=8)
    ax.tick_params(labelsize=6)
for ax in list(axes_flat)[len(region_order):]:
    ax.axis("off")

handles = [plt.Rectangle((0, 0), 1, 1, color=color_map[c]) for c in list(top_classes) + ["Other"]]
fig.legend(handles, list(top_classes) + ["Other"], loc="lower center", ncol=4, fontsize=6, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("ADS R10: DCA spatial prevalence (%) by year, within each HUC6 basin\n(each basin's own top 3 DCA, unioned, + Other; stacked bar height is a magnitude view, not a composition metric)", y=1.01)
fig.tight_layout()


As with R6, prevalence within each basin is concentrated in specific multi-year runs rather than a steady background rate -- e.g. aspen leafminer's prevalence in Tanana River and Porcupine River shows multi-year outbreak-and-decline cycles rather than a flat rate across R10's 29-year record.

### B.7 Part B limitations (DCA)

- **Multi-label pixels**: 1.98% of R10's attributed pixels carry more than one DCA in the same year -- lower than R6's 8.21%, consistent with the same "pancake" co-location pattern occurring at a lower rate here. See A.1/B.2.
- **Small-polygon dropout**: 174 of 1,196 checked small polygons (14.5%) contributed zero pixels under center-inclusion. Only 12 of these were outside all analysis subregions entirely (irrelevant to the summary); the rest are real small-polygon dropout. Full list: `outputs/qa/ads_r10_zero_pixel_polygons.csv`.
- **Reference-grid caveat**: see B.2.

### B.8 Damage Type (secondary)

In [ ]:
dmg_summary = pd.read_csv(QA_DIR / "ads_r10_damagetype_subregion_class_summary.csv")
ml_dmg_total = pd.read_csv(QA_DIR / "ads_r10_subregion_year_damagetype_multilabel_qa.csv")["attributed_pixels"].sum()
overall_dmg = dmg_summary.groupby("native_class")["pixel_count"].sum().sort_values(ascending=False)
overall_dmg_pct = (100 * overall_dmg / ml_dmg_total).round(2)

fig, ax = plt.subplots(figsize=(8, 5))
overall_dmg_pct.plot(kind="barh", ax=ax, color="#d73027")
ax.invert_yaxis()
ax.set_xlabel("spatial prevalence (%), all 20 HUC6 basins combined")
ax.set_title("ADS R10 overall DAMAGE_TYPE spatial prevalence (secondary taxonomy)")
fig.tight_layout()


Unlike R6 (where Mortality dominates), R10's damage-type signal is defoliation-led: Defoliation and its three severity-banded variants (>75%, 50-75%, <50% of leaves defoliated) together account for over 70% of attributed pixel-years, consistent with R10's DCA composition being dominated by defoliating insects (aspen leafminer, spruce beetle as a bark beetle being the notable exception). Mortality (21.9%) is a clear secondary signal. Full per-basin breakdown: `outputs/qa/ads_r10_damagetype_subregion_class_summary.csv`.

**Note:** the illustrative polygon sample (see the pointer above) and processing-history detail have been moved to **Appendix C** at the end of this notebook.

## Interpretation / Summary

### Not yet done (by design, per instruction)

- No cross-source class harmonization (native `DCA_COMMON_NAME`/`DAMAGE_TYPE` preserved as-is).
- No attempt to reconstruct ADS's original historical analysis boundaries.
- No final reference-data summary statistics, no focal-area selection.

---
# Reference Data Assessment Summary

This section summarizes what has been assessed across all four sources, without ranking them against one another or selecting a focal area/domain -- that decision is deliberately out of scope for this notebook.

### Cross-source summary table

In [ ]:
summary_rows = [
    dict(source="NCCN", primary_taxonomy="change_class", subregions=4, subregion_type="park_code",
         years="1985-2017", native_classes=16,
         attributed_pixel_years=1_130_380, derived_area_ha=101_734.2, multi_label_pct=0.12),
    dict(source="GLKN (primary, agent_01)", primary_taxonomy="agent_01", subregions=7, subregion_type="park_code",
         years="1990-2021", native_classes=10,
         attributed_pixel_years=4_077_926, derived_area_ha=367_013.3, multi_label_pct=0.00),
    dict(source="GLKN (all agents, 01+02+03)", primary_taxonomy="agent_01/02/03", subregions=7, subregion_type="park_code",
         years="1990-2021", native_classes=10,
         attributed_pixel_years=4_077_926, derived_area_ha=367_013.3, multi_label_pct=1.66),
    dict(source="ADS R6 (DCA)", primary_taxonomy="DCA_CODE", subregions=7, subregion_type="dissolved EPA L3 ecoregion",
         years="1997-2025", native_classes=91,
         attributed_pixel_years=196_383_063, derived_area_ha=17_674_475.7, multi_label_pct=8.21),
    dict(source="ADS R6 (Damage Type)", primary_taxonomy="DAMAGE_TYP", subregions=7, subregion_type="dissolved EPA L3 ecoregion",
         years="1997-2025", native_classes=16,
         attributed_pixel_years=196_383_063, derived_area_ha=17_674_475.7, multi_label_pct=3.43),
    dict(source="ADS R10 (DCA)", primary_taxonomy="DCA_COMMON_NAME", subregions=20, subregion_type="HUC6 basin",
         years="1997-2025", native_classes=67,
         attributed_pixel_years=95_700_410, derived_area_ha=8_613_036.9, multi_label_pct=1.98),
    dict(source="ADS R10 (Damage Type)", primary_taxonomy="DAMAGE_TYPE", subregions=20, subregion_type="HUC6 basin",
         years="1997-2025", native_classes=13,
         attributed_pixel_years=95_700_410, derived_area_ha=8_613_036.9, multi_label_pct=2.54),
]
pd.DataFrame(summary_rows)


### Sources assessed

Four reference-data sources, each retaining its own native attribution taxonomy: **NCCN** (NPS North Coast and Cascades Network attributed disturbance polygons, `change_class`), **GLKN** (NPS Great Lakes Network attributed disturbance data, `agent_01`/`agent_02`/`agent_03`), **ADS Region 6** (USFS Aerial Detection Survey, Pacific Northwest, `DCA_CODE`/`DAMAGE_TYP`), and **ADS Region 10** (USFS Aerial Detection Survey, Alaska, `DCA_COMMON_NAME`/`DAMAGE_TYPE`). GLKN and both ADS regions each carry two parallel attribution products (GLKN primary vs. all-agents; ADS DCA vs. Damage Type) rather than one being derived from the other.

### Subregions represented

38 analysis subregions total across the four sources: NCCN's 4 parks (MORA, NOCA, OLYM, LEWI), GLKN's 7 parks (APIS, INDU, ISRO, MISS, SACN, SLBE, VOYA), ADS R6's 7 dissolved EPA Level III ecoregions, and ADS R10's 20 HUC6 watershed basins. NCCN and GLKN subregions are the source's own `park_code` identity; **authoritative study-area AOIs now exist for both networks** (NCCN: two verified analysis-generation AOIs per park; GLKN: a per-park LandTrendr analysis-area AOI), documented as spatial/provenance context in each source's Part B.1, but not used to clip Task 1's pixel counts, which retain each source's complete published attributed geometry. ADS R6 and R10 subregions use actual boundary geometry (ecoregion / HUC6 polygons), not centroid-assigned.

### Temporal coverage

NCCN spans 1985-2017 (32 years); GLKN spans 1990-2021 (32 years); ADS R6 and ADS R10 both span 1997-2025 (29 years), reflecting ADS's status as a continuing annual survey versus NCCN/GLKN's fixed historical mapping efforts. All four are multi-decade records suitable for observing change over time, not single-snapshot datasets.

### Native attribution richness

Native class-taxonomy size varies by more than an order of magnitude across sources: GLKN's agent fields carry 10 distinct classes, NCCN's `change_class` carries 16, ADS R10's `DCA_COMMON_NAME` carries 67, and ADS R6's `DCA_CODE` carries 91. Each source's secondary/coarser taxonomy (ADS `DAMAGE_TYPE`, 13-16 classes) sits closer to NCCN/GLKN's granularity. No attempt has been made to harmonize any of these vocabularies onto a shared schema -- each is preserved and reported natively.

### Quantity of attributed reference information

Total attributed pixel-years (30 m reference grid, all subregions and years combined) range from just over 1.1 million (NCCN) to roughly 196 million (ADS R6); GLKN totals about 4.1 million and ADS R10 about 95.7 million. This spread largely reflects each source's surveyed area and survey frequency (ADS is a large-area annual aerial survey; NCCN/GLKN are park-scale efforts), not a claim about data quality or fitness for any particular downstream use. See the cross-source summary table above for the full breakdown, and each source's own Part B.3 for the per-subregion detail.

### Multi-label characteristics

Same-year, same-pixel multi-label rates differ substantially by source and by taxonomy granularity: GLKN primary (agent_01 only) is 0.00%, NCCN is 0.12%, ADS R10 DCA is 1.98%, GLKN all-agents is 1.66%, ADS R6 Damage Type is 3.43%, ADS R10 Damage Type is 2.54%, and ADS R6 DCA is 8.21% -- the highest, driven almost entirely (97% of same-year overlap, per the earlier overlap investigation) by the documented "pancake" pattern of multiple legitimate co-located observations rather than a data-quality problem. In every source, a pixel can legitimately carry more than one native-class label in the same year; spatial-prevalence percentages are reported per class and are not forced to sum to 100%.

### Major limitations / data gaps

- **Small-polygon dropout under pixel-center inclusion**: 0 (0.0%) for NCCN, 174 of 1,196 checked (14.5%) for ADS R10, and 1,373 of 2,342 checked (58.6%) for ADS R6 -- ADS R6's rate is the highest of the four, reflecting its legitimately small point/grid-cell-derived observations. GLKN had 0 polygons under the check threshold.
- **Subregion definition (`park_code`) is not spatially clipped to a boundary** for NCCN and GLKN, by design -- Task 1 retains each source's complete published attributed geometry rather than clipping to a study-area AOI. Both networks now have an authoritative one (NCCN: two verified per-generation AOIs; GLKN: the LandTrendr AOI; see Appendix A.6/B.6), and containment is high (88-100% for NCCN depending on park/generation; 99.99-100% for GLKN) -- the shortfall is attributable to legitimate boundary-straddling disturbance events (fire, defoliation, riparian change), not mis-assignment.
- **The 30 m reference grid is project-defined**, anchored to each source's native CRS origin, and has **not** been verified against the Prithvi/HLS ingestion grid -- this alignment must be revisited before these pixels are used to generate real model-training/eval samples.
- **GLKN's HUC boundary geometry is unavailable** (used only as an attribute where present, never as a spatial join), and 35.78% of confirmed GLKN disturbance area cannot be assigned to any HUC class at all -- detailed in Appendix B.
- ADS R6's raw file contains a small number of stray non-Region-6 records despite the filename (re-verified, not assumed) -- see ADS R6 Part A.

### What information is now available to support the later focal-domain decision

For all four sources, a consistent, per-subregion, per-year, per-native-class **spatial prevalence** metric is now available at 30 m resolution (native-class pixel count / unique attributed pixel count, per subregion or subregion-year), alongside the corresponding attributed pixel-year and derived-hectare totals. This gives a like-for-like (within each source's own native taxonomy) quantitative basis for comparing *how much* and *what kind* of attributed reference information exists in each subregion, over what time span, and with what multi-label/dropout caveats -- the inputs a later focal-domain selection would need. **No diversity scoring, subregion ranking, or focal-area selection has been performed here**; that is explicitly deferred to a later stage.

---
# Appendix A — NCCN: Detailed Source QA and Geography Investigation

Geometry repair, boundary-definition investigation, and other secondary implementation/debug material that informed NCCN's Part A/B (in particular, the conclusion that each park's own attributed-data footprint is used as its analysis subregion, since no complete agreed containing boundary exists for every park), and the two-generation authoritative study-area AOI investigation (A.6) that resolved that gap with new data received 2026-09-24.

## A.1 Geometry QA

Full detail: `outputs/qa/nccn_processing_report.md`. Repair method: `GeoSeries.make_valid()` (GEOS's topology-preserving repair, not the older `buffer(0)` trick).

In [ ]:
before = pd.read_csv(QA_DIR / "nccn_geometry_validity_before_repair.csv").set_index("park_code").reindex(PARKS)
after = pd.read_csv(QA_DIR / "nccn_geometry_repair_qa.csv").set_index("park_code").reindex(PARKS)

validity_table = before[["feature_count", "valid_count", "invalid_count", "invalid_pct"]]
validity_table


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(PARKS, before.loc[PARKS, "invalid_pct"], color="#d73027")
ax.set_ylabel("% invalid geometries (before repair)")
ax.set_title("NCCN geometry invalidity by park, before repair")
for i, park in enumerate(PARKS):
    ax.text(i, before.loc[park, "invalid_pct"] + 1, f"{before.loc[park, 'invalid_pct']:.1f}%", ha="center")
fig.tight_layout()


In [ ]:
# Before/after area comparison -- demonstrates make_valid() produced negligible area change
area_compare = after[["total_area_m2_before", "total_area_m2_after", "area_diff_m2", "area_pct_diff",
                        "geometry_type_changed_count", "multipart_before_count", "multipart_after_count",
                        "empty_after_repair_count", "large_change_feature_count"]]
area_compare


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
x = range(len(PARKS))
width = 0.35
ax.bar([i - width/2 for i in x], after.loc[PARKS, "total_area_m2_before"] / 1e6, width, label="before repair", color="#4575b4")
ax.bar([i + width/2 for i in x], after.loc[PARKS, "total_area_m2_after"] / 1e6, width, label="after repair", color="#91bfdb")
ax.set_xticks(list(x)); ax.set_xticklabels(PARKS)
ax.set_ylabel("total area (km²)")
ax.set_title("Total reference-polygon area before vs. after geometry repair")
ax.legend()
fig.tight_layout()
print("Max |area % diff| across all four parks:", area_compare["area_pct_diff"].abs().max(), "%")


**Result: `make_valid()` produced negligible area change** — the maximum absolute area difference across all four parks, printed above, is effectively zero (floating-point noise, ~10⁻⁷%), despite invalidity rates as high as 80% (MORA) before repair. Zero features collapsed into a mixed-type `GeometryCollection`, zero went empty, and zero of 12,630 features exceeded a 5% individual-area-change threshold (`outputs/qa/nccn_geometry_repair_large_changes.csv` — header only, no rows). This is a **measured result**, not an assumption — see `outputs/qa/nccn_processing_report.md` for the full per-feature check.

## A.2 Interactive / Visual Maps

For each park: NCCN reference polygons (red), the official NPS park boundary (black outline), and the HUC10 (blue) and HUC12 (dashed purple) watersheds currently being evaluated as possible study-area geographies. Interactive maps below are for exploratory use in Jupyter (zoom/pan, toggle layers via the control in the top-right); a static grid of the same four maps follows for report/PDF use.

In [ ]:
# MORA -- interactive map (folium). Toggle layers with the control in the top-right.
ref_park = nccn[nccn["park_code"] == "MORA"]
boundary_park = boundaries[boundaries["UNIT_CODE"] == "MORA"]
huc10_park = huc10[huc10.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]
huc12_park = huc12[huc12.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]

m = viz.interactive_overlay_map([
    (huc10_park, viz.park_layer_style("huc10"), "HUC10 watersheds", True),
    (huc12_park, viz.park_layer_style("huc12"), "HUC12 watersheds", False),
    (boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (ref_park, viz.park_layer_style("reference"), "NCCN reference polygons", True),
])
m


In [ ]:
# NOCA -- interactive map (folium). Toggle layers with the control in the top-right.
ref_park = nccn[nccn["park_code"] == "NOCA"]
boundary_park = boundaries[boundaries["UNIT_CODE"] == "NOCA"]
huc10_park = huc10[huc10.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]
huc12_park = huc12[huc12.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]

m = viz.interactive_overlay_map([
    (huc10_park, viz.park_layer_style("huc10"), "HUC10 watersheds", True),
    (huc12_park, viz.park_layer_style("huc12"), "HUC12 watersheds", False),
    (boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (ref_park, viz.park_layer_style("reference"), "NCCN reference polygons", True),
])
m


In [ ]:
# OLYM -- interactive map (folium). Toggle layers with the control in the top-right.
ref_park = nccn[nccn["park_code"] == "OLYM"]
boundary_park = boundaries[boundaries["UNIT_CODE"] == "OLYM"]
huc10_park = huc10[huc10.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]
huc12_park = huc12[huc12.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]

m = viz.interactive_overlay_map([
    (huc10_park, viz.park_layer_style("huc10"), "HUC10 watersheds", True),
    (huc12_park, viz.park_layer_style("huc12"), "HUC12 watersheds", False),
    (boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (ref_park, viz.park_layer_style("reference"), "NCCN reference polygons", True),
])
m


In [ ]:
# LEWI -- interactive map (folium). Toggle layers with the control in the top-right.
ref_park = nccn[nccn["park_code"] == "LEWI"]
boundary_park = boundaries[boundaries["UNIT_CODE"] == "LEWI"]
huc10_park = huc10[huc10.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]
huc12_park = huc12[huc12.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]

m = viz.interactive_overlay_map([
    (huc10_park, viz.park_layer_style("huc10"), "HUC10 watersheds", True),
    (huc12_park, viz.park_layer_style("huc12"), "HUC12 watersheds", False),
    (boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (ref_park, viz.park_layer_style("reference"), "NCCN reference polygons", True),
])
m


In [ ]:
# Static equivalent of the four interactive maps above, for HTML/PDF export
fig, axes = plt.subplots(2, 2, figsize=(13, 13))
for ax, park in zip(axes.flat, PARKS):
    ref_park = nccn[nccn["park_code"] == park]
    boundary_park = boundaries[boundaries["UNIT_CODE"] == park]
    huc10_park = huc10[huc10.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]
    huc12_park = huc12[huc12.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]
    viz.static_overlay_map([
        (huc10_park, viz.park_layer_style_mpl("huc10"), "HUC10 watersheds"),
        (huc12_park, viz.park_layer_style_mpl("huc12"), "HUC12 watersheds"),
        (boundary_park, viz.park_layer_style_mpl("park_boundary"), "NPS park boundary"),
        (ref_park, viz.park_layer_style_mpl("reference"), "NCCN reference polygons"),
    ], title=f"{park} -- {PARK_NAMES[park]}", ax=ax)
fig.suptitle("NCCN reference polygons, NPS park boundary, and HUC10/HUC12 watersheds", y=1.01, fontsize=13)
fig.tight_layout()


## A.3 Park-Boundary Comparison

Measured in `src/process_nccn.py` Step 5, full numbers in `outputs/qa/nccn_boundary_relationship_qa.csv`.

In [ ]:
bqa = pd.read_csv(QA_DIR / "nccn_boundary_relationship_qa.csv").set_index("park_code").reindex(PARKS)
bqa[["total_reference_polygons", "entirely_inside_count", "straddling_boundary_count",
     "entirely_outside_count", "pct_area_inside", "pct_area_outside"]]


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(PARKS, bqa.loc[PARKS, "pct_area_inside"], label="% area inside park", color="#1a9850")
ax.bar(PARKS, bqa.loc[PARKS, "pct_area_outside"], bottom=bqa.loc[PARKS, "pct_area_inside"],
       label="% area outside park", color="#d73027")
ax.set_ylabel("% of total reference area")
ax.set_title("NCCN reference-polygon area: inside vs. outside the NPS park boundary")
ax.legend(loc="lower right")
for i, park in enumerate(PARKS):
    ax.text(i, 101, f"{bqa.loc[park, 'pct_area_inside']:.1f}% in", ha="center", fontsize=8)
fig.tight_layout()


In [ ]:
# Maps colored by inside / straddling / outside the park boundary -- makes the relationship
# visually unambiguous rather than relying on the numbers alone.
fig, axes = plt.subplots(2, 2, figsize=(13, 13))
for ax, park in zip(axes.flat, PARKS):
    ref_park = nccn[nccn["park_code"] == park]
    boundary_geom = boundaries.loc[boundaries["UNIT_CODE"] == park, "geometry"].iloc[0]
    viz.static_inside_outside_map(
        ref_park, boundary_geom,
        title=f"{park} -- inside/outside park boundary ({bqa.loc[park, 'pct_area_inside']:.1f}% inside)",
        ax=ax,
    )
fig.suptitle("NCCN reference polygons colored by relationship to the NPS park boundary", y=1.01, fontsize=13)
fig.tight_layout()


**Measured result:** MORA (4.5% inside), NOCA (10.4% inside), and LEWI (0.9% inside) all have the large majority of their reference-polygon area *outside* the strict NPS park boundary. OLYM is the exception (86.4% inside). This means the NPS park boundary alone is not an appropriate sole summarization geography for NCCN — see Section 7.

## A.4 HUC Exploratory Test — HUC10 and HUC12

Full report: `outputs/qa/nccn_huc_fit_test.md`. **Important — these are separate findings, not one:** (1) the reference data is almost entirely *contained within* the union of a set of HUC watersheds at both resolutions tested; (2) the reference data's actual *edge* essentially does not align with HUC boundary lines, at either resolution. Finding (1) alone would wrongly suggest HUCs explain the study area; finding (2) is why they don't. **Neither HUC10 nor HUC12 is being presented as the NCCN study-area geography** — this section tests that hypothesis at two resolutions, and it is not supported by the edge-alignment evidence at either one.

HUC10 (117 features) and HUC12 (469 features) were both exported by the user into `data/raw/boundaries/nps/Prithvi_NCCN/`, and are run through the identical analysis method (`src/nccn_huc_fit_test.py`) so the two resolutions are directly comparable — the HUC10 logic itself was not redone, only generalized to also accept HUC12.

In [ ]:
hqa10 = pd.read_csv(QA_DIR / "nccn_huc10_fit_summary.csv").set_index("park_code").reindex(PARKS)
hqa12 = pd.read_csv(QA_DIR / "nccn_huc12_fit_summary.csv").set_index("park_code").reindex(PARKS)
comparison = pd.read_csv(QA_DIR / "nccn_huc10_vs_huc12_comparison.csv").set_index("park_code").reindex(PARKS)

comparison[["n_huc_intersecting_huc10", "pct_area_within_selected_hucs_huc10", "edge_alignment_fraction_huc10",
            "n_huc_intersecting_huc12", "pct_area_within_selected_hucs_huc12", "edge_alignment_fraction_huc12"]]


In [ ]:
# Coverage vs. edge alignment, HUC10 vs HUC12 side by side -- the key comparison.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
x = range(len(PARKS))
width = 0.35

ax1.bar([i - width/2 for i in x], comparison.loc[PARKS, "pct_area_within_selected_hucs_huc10"], width,
        label="HUC10", color="#4575b4")
ax1.bar([i + width/2 for i in x], comparison.loc[PARKS, "pct_area_within_selected_hucs_huc12"], width,
        label="HUC12", color="#984ea3")
ax1.set_xticks(list(x)); ax1.set_xticklabels(PARKS)
ax1.set_ylabel("% of reference area")
ax1.set_title("Coverage: reference area inside\nthe union of intersecting HUCs")
ax1.set_ylim(0, 110)
ax1.legend()

ax2.bar([i - width/2 for i in x], comparison.loc[PARKS, "edge_alignment_fraction_huc10"] * 100, width,
        label="HUC10", color="#4575b4")
ax2.bar([i + width/2 for i in x], comparison.loc[PARKS, "edge_alignment_fraction_huc12"] * 100, width,
        label="HUC12", color="#984ea3")
ax2.set_xticks(list(x)); ax2.set_xticklabels(PARKS)
ax2.set_ylabel("% of footprint boundary length")
ax2.set_title("Edge alignment: reference-footprint edge\nwithin 100m of a HUC boundary line")
ax2.set_ylim(0, 10)
ax2.legend()

fig.suptitle("High coverage at both resolutions (left) vs. near-zero edge alignment at both (right)", y=1.03)
fig.tight_layout()
print("HUC12 shows no meaningful improvement in edge alignment over HUC10, despite 3-4x more units per park.")


### Nesting check: are the selected HUC12s a coherent subset of the selected HUC10s?

In [ ]:
nesting = pd.read_csv(QA_DIR / "nccn_huc10_huc12_nesting.csv").set_index("park_code").reindex(PARKS)
nesting[["n_huc12_selected", "n_distinct_huc10_parents_of_selected_huc12",
         "n_huc10_selected_independently", "nesting_is_clean"]]


**Result: yes, cleanly** — for every park, the distinct HUC10 parents implied by the selected HUC12s (via the standard 10-digit-prefix convention) exactly match the HUC10s independently found to intersect the same reference data. This confirms the two HUC exports are internally consistent with each other. **It is not, on its own, evidence for the watershed hypothesis** — it only shows the two layers agree geographically, which they should regardless of whether HUCs explain anything about NCCN's actual monitoring extent.

**On whether selecting HUC12s by intersection creates an artificial, disturbance-chasing boundary** (cautious interpretation, not a measured fact): going from HUC10 to HUC12, the number of intersecting units roughly triples-to-quadruples per park while the *aggregate* fraction of the selected HUC12 area that's actually reference data stays low (1.2%–9.1%, only modestly higher than HUC10's 0.9%–6.5% — an increase attributable to smaller unit size, not better fit, since a small patch wastes proportionally less of a small HUC12 than of a large HUC10). This is consistent with, though not proof of, the selection criterion tracking scattered disturbance locations rather than reconstructing a real finer-grained boundary. Full numbers in `outputs/qa/nccn_huc_fit_test.md`.

In [ ]:
# Static maps showing HUC10 and HUC12 together against the reference data --
# lets the nesting/edge relationship be inspected visually, not just numerically.
fig, axes = plt.subplots(2, 2, figsize=(13, 13))
for ax, park in zip(axes.flat, PARKS):
    ref_park = nccn[nccn["park_code"] == park]
    huc10_park = huc10[huc10.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]
    huc12_park = huc12[huc12.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]
    viz.static_overlay_map([
        (huc10_park, viz.park_layer_style_mpl("huc10"), "HUC10 watersheds"),
        (huc12_park, viz.park_layer_style_mpl("huc12"), "HUC12 watersheds"),
        (ref_park, viz.park_layer_style_mpl("reference"), "NCCN reference polygons"),
    ], title=f"{park} -- HUC10 (solid blue) vs HUC12 (dashed purple)", ax=ax)
fig.suptitle("HUC10/HUC12 nesting and edge relationship against NCCN reference polygons", y=1.01, fontsize=13)
fig.tight_layout()


In [ ]:
per_huc10 = pd.read_csv(QA_DIR / "nccn_huc10_fit_per_huc.csv")
per_huc12 = pd.read_csv(QA_DIR / "nccn_huc12_fit_per_huc.csv")
print("Full per-HUC10 breakdown (all parks):")
per_huc10.sort_values(["park_code", "ref_area_within_huc_m2"], ascending=[True, False])[
    ["park_code", "huc_id", "name", "pct_of_park_ref_area", "pct_of_huc_filled"]
]


In [ ]:
print("Full per-HUC12 breakdown (all parks, top contributor per park shown; full table in the CSV):")
per_huc12.sort_values(["park_code", "ref_area_within_huc_m2"], ascending=[True, False]).groupby("park_code").head(5)[
    ["park_code", "huc_id", "name", "pct_of_park_ref_area", "pct_of_huc_filled"]
]


**Data-quality flag:** NOCA's HUC12 coverage (93.13%) is measurably lower than its HUC10 coverage (99.86%) for the *same* reference data — unexpected, since HUC12 watersheds should tile all land just as exhaustively as HUC10. This is flagged as a possible gap in the HUC12 export for NOCA specifically, not a finding about the monitoring geography (see `outputs/qa/nccn_huc_fit_test.md` for detail — not resolved here).

**Measured result, both resolutions:** ~93–100% coverage in every park at both HUC10 and HUC12, but edge alignment of 0.000–0.021 (HUC10) and 0.001–0.021 (HUC12) — essentially zero at both. **Interpretation (not a measured fact):** this pattern is consistent with the reference data's extent being drawn independently of watershed boundaries at any resolution tested, with the high coverage number being close to a mathematical inevitability rather than evidence of intent.

## A.5 Interpretation / Open Questions

Distinguishing measured results, documented source statements, and our own interpretation — none of the interpretation items below should be treated as settled.

### Measured (this notebook / `src/process_nccn.py` / `src/nccn_huc_fit_test.py`)

- Geometry repair via `make_valid()` produced negligible area change (Section 3) across all four parks, despite up to 80% pre-repair invalidity.
- The NPS park boundary contains only 4.5% (MORA), 10.4% (NOCA), 86.4% (OLYM), and 0.9% (LEWI) of each park's total reference-polygon area (Section 5).
- The union of intersecting HUC10 watersheds contains ~93–100% of each park's reference area, but the reference-data footprint's edge shows essentially zero alignment (0.000–0.021) with HUC10 boundary lines (Section 6).
- **The same test repeated at HUC12 resolution (469 units vs. HUC10's 117) shows the same result**: ~93–100% coverage, edge alignment 0.001–0.021 — no meaningful improvement over HUC10, despite 3-4x more units per park (Section 6).
- HUC10 and HUC12 nest cleanly and consistently with each other in all four parks (Section 6).

### Documented (NCCN source certification forms — see `docs/data_inventory.md` §2.3)

- NCCN's own certification forms state that monitoring was conducted for the park **and surrounding USFS Wilderness**, within a "Protected Areas" study area — an administrative/ecological boundary, not a boundary the forms describe as hydrological.
- At least one MORA cert form notes 31 fire patches were deliberately **not clipped** to the study area, to preserve full event extent.
- NCCN developed the landscape-change protocol as part of NPS Vital Signs Monitoring; initial implementations were NOCA (2012), MORA (2013), OLYM (2014), using OSU/eMapR LandTrendr (Section 1). This describes patch generation/classification, not the study-area boundary.

### Interpretation / open questions (not yet settled — do not treat as conclusions)

- **NPS administrative boundaries alone do not represent the NCCN reference-data extent** for three of four parks. This is directly supported by the measured Section 5 results.
- **Neither HUC10 nor HUC12 boundaries appear to explain the NCCN study-area shape.** Supported by the Section 6 edge-alignment result at both resolutions — testing a finer HUC level did not reveal a boundary pattern HUC10 was too coarse to show. Spatial fit (even high coverage) does not by itself establish that NCCN organized its monitoring around watershed boundaries, and the evidence here argues against that hypothesis at both tested resolutions.
- NCCN documentation refers to a broader "Protected Areas" monitoring/study geography that is neither the park boundary nor (apparently) a HUC10 or HUC12 watershed grouping.
- **The exact, reproducible boundary for that "Protected Areas" geography has not yet been obtained or tested.** Until it is, no NCCN region-level summary should be computed against the park boundary or either HUC resolution without explicitly deciding how to handle the excluded/misaligned area.
- NOCA's HUC12 coverage (93.13%) is measurably lower than its HUC10 coverage (99.86%) — flagged as a possible HUC12 export gap for NOCA, not yet resolved.

### 7.1 Investigating the NOCA HUC12 coverage gap

**Question:** HUC10 covers 99.86% of NOCA's reference area, but the exported HUC12 selection covers only 93.13% of the *same* reference data (Section 6). Is this an NCCN-geography finding, or an input-completeness problem with the HUC12 export? This is primarily a QA question about our own inputs, not a finding about NCCN's monitoring extent -- treated as such below.

In [ ]:
# Where is the NOCA reference area that HUC12 misses, and is it covered by HUC10?
noca = nccn[nccn["park_code"] == "NOCA"].copy()
h10_sel_noca = huc10[huc10.geometry.apply(lambda h: noca.geometry.intersects(h).any())]
h12_sel_noca = huc12[huc12.geometry.apply(lambda h: noca.geometry.intersects(h).any())]
h10_union_noca = h10_sel_noca.geometry.union_all()
h12_union_noca = h12_sel_noca.geometry.union_all()

noca["area_missing_from_huc12"] = noca.geometry.area - noca.geometry.intersection(h12_union_noca).area
missing = noca[noca["area_missing_from_huc12"] > 1].copy()  # > 1 sq m, i.e. ignore float noise

total_missing = missing["area_missing_from_huc12"].sum()
covered_by_huc10 = missing.geometry.intersection(h10_union_noca).area.sum()

print(f"{len(missing)} of {len(noca)} NOCA reference polygons have area outside the HUC12 selection")
print(f"Total missing area: {total_missing:,.0f} m2 ({total_missing/1e4:,.1f} ha)")
pct_covered = min(100 * covered_by_huc10 / total_missing, 100.0)
print(f"Of that missing area, essentially all of it ({pct_covered:.1f}%, capped at 100 -- "
      f"raw ratio can exceed 100% by a sliver from independent-union edge overlap) IS covered by HUC10 --")
print("i.e. it sits inside an already-selected HUC10, just not inside any selected HUC12.")
print("This points directly at the HUC12 export being incomplete, not at a real geographic difference.")


In [ ]:
# Pinpoint the specific gap: which HUC10 accounts for nearly all the missing area,
# and does it have a literal missing child HUC12?
missing_centroids = missing.geometry.centroid
by_huc10 = {}
for _, h in h10_sel_noca.iterrows():
    inside = missing_centroids.within(h.geometry)
    if inside.sum() > 0:
        by_huc10[h["huc10"]] = (h["name"], int(inside.sum()), missing.loc[inside[inside].index, "area_missing_from_huc12"].sum())

print("HUC10s containing the polygons missing from HUC12 coverage:")
for k, (name, n, area) in sorted(by_huc10.items(), key=lambda kv: -kv[1][2]):
    print(f"  {k} ({name}): {n} polygons, {area:,.0f} m2 missing")

parent = max(by_huc10, key=lambda k: by_huc10[k][2])
h10geom = huc10.loc[huc10["huc10"] == parent, "geometry"].iloc[0]
children = huc12[huc12["huc12"].str.startswith(parent)]
print(f"\nExported HUC12 children of {parent} ({by_huc10[parent][0]}):")
print(children[["huc12", "name"]].to_string(index=False))

child_union = children.geometry.union_all()
gap_geom = h10geom.difference(child_union)
print(f"\nGap = this HUC10's area not covered by ANY exported child HUC12: "
      f"{gap_geom.area:,.0f} m2 ({100*gap_geom.area/h10geom.area:.1f}% of this HUC10's area)")
print("Note the child code sequence above -- a numbering gap (e.g. ...01,02,03,04,06 with no 05)")
print("is direct evidence a HUC12 sub-unit is simply absent from the export, not a real spatial gap.")


In [ ]:
# Show the gap on a map: the HUC10 (blue), the exported HUC12 children (dashed purple),
# the un-exported gap area (orange), and the reference polygons missing HUC12 coverage (black).
fig, ax = plt.subplots(figsize=(8, 8))
viz.static_overlay_map([
    (gpd.GeoDataFrame(geometry=[h10geom], crs=nccn.crs), viz.park_layer_style_mpl("huc10"), "HUC10 (Upper Lake Chelan)"),
    (children, viz.park_layer_style_mpl("huc12"), "Exported HUC12 children"),
    (gpd.GeoDataFrame(geometry=[gap_geom], crs=nccn.crs),
     dict(facecolor="#fdae61", edgecolor="none", alpha=0.55), "Gap: no exported HUC12 here"),
    (missing, dict(facecolor="#000000", edgecolor="none"), "NCCN reference polygons missing HUC12 coverage"),
], title="NOCA HUC12 export gap -- Upper Lake Chelan (1702000902)", ax=ax)
fig.tight_layout()


**Conclusion: input-completeness issue, not an NCCN geography finding.** Essentially all (>99%) of the area missing from the HUC12 selection sits inside an already-selected HUC10 (Upper Lake Chelan, `1702000902`), and the exported HUC12 children for that HUC10 leave a directly-measured 43.7% gap in area with a literal missing code in the sequence. This is consistent with the HUC12 export (an Earth Engine / GIS export the user performed) simply not including every HUC12 required to fully cover the already-correctly-selected HUC10 set — not a difference in the underlying NCCN monitoring geography. Recommend re-exporting NOCA's HUC12 selection to include the missing unit(s) before treating NOCA's HUC12 numbers as final; this does not change the Section 6 conclusion (edge alignment is unaffected — the gap is an interior hole, not an edge-alignment artifact).

### 7.2 Investigating the documented NCCN "Protected Areas" geography

Re-examined every NCCN source document in `docs/source_docs/nccn/` for language defining the study/monitoring extent — not just the two documents read in the first inspection pass (MORA, LEWI), but also NOCA's and OLYM's current-schema (V2.1.1) certification forms and both legacy (V2B) certification forms, none of which had been read in full before this. Quoted verbatim below, not paraphrased.

**MORA (V2.1.1 certification form):**
> "Landsat/LandTrendr derived landscape change data from Mount Rainier National Park **and surrounding USFS Wilderness areas ("Protected Areas" study area)**..."
> "Verified that polygon centroid coordinates were within the 'Protected Areas' study area **polygon** for Mount Rainier National Park;"

**NOCA (V2.1.1 certification form) — identical construction:**
> "...North Cascades National Park Complex (NOCA) **and surrounding USFS Wilderness areas ("Protected Areas" study area)**..."

**OLYM (V2.1.1 certification form) — identical construction:**
> "...Olympic National Park **and surrounding USFS Wilderness areas ("Protected Areas" study area)**..."

**LEWI (certification form) — notably different, no Wilderness reference at all:**
> "Landsat/LandTrendr derived landscape change data from Lewis and Clark National Historical Park **and surrounding study area**..." (no mention of USFS Wilderness anywhere in this document)

**Legacy V2B forms (NOCA, OLYM)** use a third, slightly different phrasing in the headline description line — "Park X **and surrounding study area**" (dropping "USFS Wilderness" from that specific sentence), though "USFS wilderness areas" reappears later in the same documents' QA-check bullet lists. A minor internal inconsistency, not a substantive difference from the current V2.1.1 wording.

**What the documentation does *not* say, checked directly (zero hits for "buffer" in any of the six documents searched):**
- No named specific Wilderness area (e.g. no "Norse Peak Wilderness," "William O. Douglas Wilderness," etc.) for any park.
- No buffer distance or other explicit spatial construction rule.
- No cited source dataset for "USFS Wilderness areas" (no PAD-US, no USFS wilderness layer name/vintage).

**What the documentation does establish:** a specific "Protected Areas study area **polygon**" already exists and was used operationally by NCCN during their own QA ("verified that polygon centroid coordinates were within the... study area polygon") — this is referenced as an existing GIS object, not something to be freshly constructed from general rules.

**A concrete lead:** MORA, NOCA, and OLYM's V2.1.1 forms all cite the *same* single combined reference: *"NCCN, Antonova N., Copass C. 2022. NCCN landscape change monitoring polygons in and around Mount Rainier, North Cascades, and Olympic National Parks for 1987-2017"*, with an NPS IRMA DataStore link: **https://irma.nps.gov/DataStore/Reference/Profile/2294375**. This was not fetched or checked in this pass (that would require web access, not exercised here without being asked) — it is the most promising next step for finding the actual "Protected Areas" boundary or a report that defines it precisely. LEWI cites a separate, different report ("Landsat-based monitoring of landscape change in Lewis and Clark National Historical Park: 1985–2011," NRDS NPS/NCCN/NRDS—2019/1206) with no URL in our files.

#### Classification

**C — documentation describes the geography but is insufficient to reproduce it exactly**, for MORA, NOCA, and OLYM: we know the study area is "the park plus surrounding USFS Wilderness areas," and that a real, specific GIS polygon implementing this already exists and was used by NCCN — but without knowing *which* Wilderness units, what adjacency/buffer rule (if any) was applied, or the exact source Wilderness boundary dataset and vintage, we cannot reconstruct the exact polygon from documented rules and generic authoritative boundary data (which would be classification B). Approximating it ourselves (buffer, convex hull, bounding box, or a selected-HUC union) was explicitly out of scope for this investigation and has not been done.

**D — for LEWI specifically**, the documentation is even thinner (no Wilderness reference at all, just "surrounding study area"), and the cited report is not in our files at all — LEWI needs the original study-area boundary, or at minimum its cited 2019 NRDS report, from NCCN/Natasha more than the other three do.

**Recommended next step:** request the actual "Protected Areas" study-area polygon(s) directly from NCCN/Natasha, since the documentation confirms such a polygon already exists and is used operationally — reconstructing it independently is not well-supported by what's documented. Checking the IRMA DataStore reference above is a reasonable parallel step if web access is available.

### Next steps (not started)

- Attempt to obtain or reconstruct the actual "Protected Areas" study-area boundary referenced in the certification forms — this remains the most direct path to a real NCCN summarization geography, now that both HUC resolutions have been tested and neither is supported.
- Check the NOCA HUC12 coverage gap against the source WBD before relying on that park's HUC12 numbers specifically.
- Extend this notebook with GLKN, ADS R6, ADS R10, and cross-source sections once each source reaches the same processing/QA stage as NCCN.

### A.6 Authoritative study-area AOI investigation (added 2026-09-24)

Natasha Antonova (NCCN's data provider) supplied both generations of NCCN's study-area boundary (`data/raw/boundaries/nps/Prithvi_NCCN/`), confirming the two-era history already documented in A.1/A.2:

- **Original analysis (1985-2009/10/11): 10-mile buffer.** `LPa01_LEWI_MORA_NOCA_OLYM.shp` -- one dissolved polygon per park (LEWI split N/S), `CREATED_BY`="N. Antonova", `SOURCE`="NCCN Landscape Dynamics Monitoring Protocol".
- **Later analysis (1987-2017): "Protected Areas" (NPS + USFS Wilderness).** `{MORA,NOCA,OLYM}_USFS_NPS_StudyArea.shp` -- parcel-level (24-55 features/park). **No LEWI file exists in this set**, consistent with the later, narrower extent excluding areas NPS did not intend to track private-property change on.

**Verified generation mapping** (not assumed) -- every currently-used Task 1 dataset checked against BOTH AOI generations, plus the two superseded legacy datasets for confirmation (`src/qa_nccn_aoi_generation_mapping.py`, `outputs/qa/nccn_aoi_generation_mapping_qa.csv`):

In [ ]:
gen_map = pd.read_csv(QA_DIR / "nccn_aoi_generation_mapping_qa.csv")
cols = ["dataset", "park", "feature_count", "total_area_ha", "year_range", "generation", "used_in_task1",
        "pct_area_in_lpa01_10mi_buffer", "pct_area_in_protected_areas",
        "n_centroid_outside_lpa01_10mi_buffer", "n_centroid_outside_protected_areas"]
gen_map[cols]


Every currently-used dataset fits its OWN proper-generation AOI at 88-100% area with 0-1 negligible (4.2-19.4 m) centroid-outside cases -- confirming Natasha's expected mapping exactly (MORA/NOCA/OLYM V2.1.1 -> Protected Areas; LEWI -> LPa01). The 2 superseded legacy datasets (`NOCA_1985_2009_V2B`, `OLYM_1985_2010_V2B`) fit their proper (original/10-mile-buffer) generation at 100% area with zero genuine outliers, and are overwhelmingly, genuinely outside Protected Areas (6,508 of 9,980 and 20,877 of 22,553 polygons respectively, up to 39 km away) -- independent confirmation, via AOI geometry rather than just filenames/dates, that they belong to the earlier generation and are correctly excluded from Task 1.

**The apparent NOCA anomaly (fitting the narrower Protected Areas AOI, 98.12%, better than the wider LPa01 AOI, 88.24%) is a cross-generation comparison artifact, not a data problem, and disappears under proper pairing.** Checking NOCA V2.1.1 against the *wrong*-generation LPa01 shows 476 of 518 non-contained polygons with a genuinely-outside centroid, up to 13.5 km away -- a real mismatch. That vanishes (to 1 negligible 4.2 m case) once NOCA is checked only against its own proper AOI (Protected Areas). Root cause: NOCA's Protected Areas AOI is only 96.34% geometrically nested inside its LPa01 AOI (vs. 100% for MORA/OLYM) -- North Cascades' more spatially complex protected-lands footprint (Ross Lake/Lake Chelan NRA parcels, Canadian border adjacency) extends beyond the 2013-drawn 10-mile buffer in places MORA's/OLYM's simpler footprints don't.

**Method caveat:** an initial pass used Hausdorff distance between each polygon's outside-portion and the AOI as a "how far outside" metric, and produced spurious tens-of-km values for polygons a direct centroid check confirmed were 0 m outside -- a GEOS artifact on the difference() of complex multi-part (24-55 parcel) unioned geometries, not a real excursion. The corrected metric used throughout (centroid-distance-to-AOI, % of each polygon's own area outside) is documented in `src/qa_nccn_aoi_generation_mapping.py`.

---
# Appendix B — GLKN: Detailed Source QA and Geography Investigation

Geometry repair, HUC-geography QA, and other secondary implementation/debug material that informed GLKN's Part A/B (in particular, why HUC boundaries were investigated but not adopted as the subregion definition, and the Isle Royale/Voyageurs Canada custom-HUC-code investigation, now resolved by the authoritative LandTrendr AOI, B.6).

## B.1 Geometry QA

Full detail: `outputs/qa/glkn_processing_report.md`. Same method as NCCN: `GeoSeries.make_valid()`, tested on confirmed disturbance polygons only.

In [ ]:
glkn_before = pd.read_csv(QA_DIR / "glkn_geometry_validity_before_repair.csv")
glkn_before["park_code"] = glkn_before["park_code"].str.upper()  # source CSV uses lowercase; GLKN_PARKS is uppercase
glkn_before = glkn_before.set_index("park_code").reindex(GLKN_PARKS)

glkn_after = pd.read_csv(QA_DIR / "glkn_geometry_repair_qa.csv")
glkn_after["park_code"] = glkn_after["park_code"].str.upper()
glkn_after = glkn_after.set_index("park_code").reindex(GLKN_PARKS)

glkn_before[["feature_count", "valid_count", "invalid_count", "invalid_pct"]]


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(GLKN_PARKS, glkn_before.loc[GLKN_PARKS, "invalid_pct"], color="#d73027")
ax.set_ylabel("% invalid geometries (before repair)")
ax.set_title("GLKN geometry invalidity by park, before repair\n(compare to NCCN: up to 80% -- GLKN is far healthier)")
for i, park in enumerate(GLKN_PARKS):
    ax.text(i, glkn_before.loc[park, "invalid_pct"] + 0.15, f"{glkn_before.loc[park, 'invalid_pct']:.2f}%", ha="center", fontsize=8)
fig.tight_layout()


In [ ]:
glkn_area_compare = glkn_after[["total_area_m2_before", "total_area_m2_after", "area_diff_m2", "area_pct_diff",
                                  "geometry_type_changed_count", "empty_after_repair_count", "large_change_feature_count"]]
glkn_area_compare


**Result: `make_valid()` produced negligible area change** — max |area % diff| across all 7 parks is 0.0% (floating-point noise), zero empty geometries, zero features exceeding a 5% individual-area-change threshold across all 53,665 confirmed polygons. No stop condition was triggered (invalidity itself was also far lower than NCCN's worst cases: 0.27%–4.43% vs. up to 80%).

## B.2 HUC Geography QA

Using confirmed disturbance rows only. **No HUC boundary geometry exists for GLKN yet** (only the per-polygon `HUC_12` attribute) — this is attribute-level QA, not a map of HUC boundaries.

In [ ]:
huc_geog = pd.read_csv(QA_DIR / "glkn_huc_geography.csv")
huc_class_by_park = pd.crosstab(huc_geog["park_code"], huc_geog["huc12_class"],
                                  values=huc_geog["polygon_count"], aggfunc="sum").reindex(GLKN_PARKS).fillna(0).astype(int)
huc_class_by_park


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
huc_class_by_park.plot(kind="bar", stacked=True, ax=ax, color={"standard": "#4575b4", "nonstandard": "#d73027"})
ax.set_ylabel("confirmed polygons")
ax.set_title("Standard-format vs. non-standard HUC_12 codes, by park")
ax.legend(title="")
fig.tight_layout()


**Result: 35.78% of confirmed disturbance area (31.58% of polygons) cannot be assigned to a standard-derived HUC10**, almost entirely because **99.9% of ISRO's confirmed rows (16,023 of 16,042) use the non-standard Isle Royale code scheme** (`2AA-01` style — undocumented in either source document, see `docs/data_inventory.md` §3.5 Q5). VOYA is affected to a smaller degree (14.8% of its rows). **No HUC10 was invented for these rows** — they are flagged in `outputs/qa/glkn_huc10_unassigned.csv`, not silently assigned or dropped. Any future HUC10-level GLKN summary must address this explicitly, since a naive approach would effectively remove Isle Royale from the analysis.

## B.3 Interactive / Visual Maps

For each of the seven GLKN park/monitoring landscapes, independently viewable: confirmed disturbance polygons (red) and the official NPS park boundary (black outline) for context. **No HUC layer is shown** (not yet available for GLKN — see Section 4). As with NCCN, interactive maps are for exploratory use in Jupyter; a static grid follows for the eventual HTML/PDF report. The maps make it visually obvious that GLKN reference data extends well beyond the formal park boundary in several parks, exactly as documented.

In [ ]:
# APIS (Apostle Islands National Lakeshore) -- interactive map
glkn_park = glkn[glkn["park_code"] == "APIS"]
glkn_boundary_park = glkn_boundaries[glkn_boundaries["UNIT_CODE"] == "APIS"]

m = viz.interactive_overlay_map([
    (glkn_boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (glkn_park, viz.park_layer_style("reference"), "GLKN confirmed disturbance polygons", True),
])
m


In [ ]:
# INDU (Indiana Dunes National Park) -- interactive map
glkn_park = glkn[glkn["park_code"] == "INDU"]
glkn_boundary_park = glkn_boundaries[glkn_boundaries["UNIT_CODE"] == "INDU"]

m = viz.interactive_overlay_map([
    (glkn_boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (glkn_park, viz.park_layer_style("reference"), "GLKN confirmed disturbance polygons", True),
])
m


In [ ]:
# ISRO (Isle Royale National Park) -- interactive map
glkn_park = glkn[glkn["park_code"] == "ISRO"]
glkn_boundary_park = glkn_boundaries[glkn_boundaries["UNIT_CODE"] == "ISRO"]

m = viz.interactive_overlay_map([
    (glkn_boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (glkn_park, viz.park_layer_style("reference"), "GLKN confirmed disturbance polygons", True),
])
m


In [ ]:
# MISS (Mississippi National River and Recreation Area) -- interactive map
glkn_park = glkn[glkn["park_code"] == "MISS"]
glkn_boundary_park = glkn_boundaries[glkn_boundaries["UNIT_CODE"] == "MISS"]

m = viz.interactive_overlay_map([
    (glkn_boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (glkn_park, viz.park_layer_style("reference"), "GLKN confirmed disturbance polygons", True),
])
m


In [ ]:
# SACN (Saint Croix National Scenic Riverway) -- interactive map
glkn_park = glkn[glkn["park_code"] == "SACN"]
glkn_boundary_park = glkn_boundaries[glkn_boundaries["UNIT_CODE"] == "SACN"]

m = viz.interactive_overlay_map([
    (glkn_boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (glkn_park, viz.park_layer_style("reference"), "GLKN confirmed disturbance polygons", True),
])
m


In [ ]:
# SLBE (Sleeping Bear Dunes National Lakeshore) -- interactive map
glkn_park = glkn[glkn["park_code"] == "SLBE"]
glkn_boundary_park = glkn_boundaries[glkn_boundaries["UNIT_CODE"] == "SLBE"]

m = viz.interactive_overlay_map([
    (glkn_boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (glkn_park, viz.park_layer_style("reference"), "GLKN confirmed disturbance polygons", True),
])
m


In [ ]:
# VOYA (Voyageurs National Park) -- interactive map
glkn_park = glkn[glkn["park_code"] == "VOYA"]
glkn_boundary_park = glkn_boundaries[glkn_boundaries["UNIT_CODE"] == "VOYA"]

m = viz.interactive_overlay_map([
    (glkn_boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (glkn_park, viz.park_layer_style("reference"), "GLKN confirmed disturbance polygons", True),
])
m


In [ ]:
# Static equivalent of the seven interactive maps above, for HTML/PDF export
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
for ax, park in zip(axes.flat, GLKN_PARKS):
    glkn_park = glkn[glkn["park_code"] == park]
    glkn_boundary_park = glkn_boundaries[glkn_boundaries["UNIT_CODE"] == park]
    viz.static_overlay_map([
        (glkn_boundary_park, viz.park_layer_style_mpl("park_boundary"), "NPS park boundary"),
        (glkn_park, viz.park_layer_style_mpl("reference"), "GLKN confirmed disturbance polygons"),
    ], title=f"{park} -- {GLKN_PARK_NAMES[park]}", ax=ax)
axes.flat[-1].axis("off")
fig.suptitle("GLKN confirmed disturbance polygons vs. NPS park boundary, all 7 parks", y=1.01, fontsize=13)
fig.tight_layout()


## B.4 Key QA Questions

Full numbers: `outputs/qa/glkn_processing_report.md` §Step 6.

In [ ]:
# Q1: multiple agents populated?
n_agents = glkn[["agent_01", "agent_02", "agent_03"]].notna().sum(axis=1)
print("Number of agents populated per confirmed polygon:")
print(n_agents.value_counts().sort_index())
print(f"\n{(n_agents>1).sum()} of {len(glkn)} confirmed polygons ({100*(n_agents>1).sum()/len(glkn):.2f}%) have more than one agent.")


In [ ]:
# Q2: uniqID repeat pattern -- polygon count vs. distinct physical patch count.
# NOTE: polygon counts are NOT called "event counts" or "patch counts" here or
# anywhere else in this notebook, per the source documentation's own caveat
# (individual disturbance events can span multiple polygons) and this repeat pattern.
uid_counts = glkn["uniqID"].value_counts()
print(f"Distinct uniqID values: {glkn['uniqID'].nunique():,} across {len(glkn):,} confirmed polygon rows")
print(f"-> raw polygon counts overstate distinct uniqID groups by "
      f"{100*(len(glkn)-glkn['uniqID'].nunique())/glkn['uniqID'].nunique():.1f}%")
print("\nRepeat-count distribution (how many uniqIDs appear 1x, 2x, 3x...):")
print(uid_counts.value_counts().sort_index())


In [ ]:
# Q3: confirmed disturbance area and counts by year and park (summary table; full detail is
# the year-histogram grid in Section 2)
by_year_park = glkn.groupby(["park_code", "year"]).agg(
    polygon_count=("source_feature_id", "count"),
    area_ha=("geometry", lambda g: g.area.sum() / 1e4),
).reset_index()
print(f"{len(by_year_park)} park-year combinations. Example (SLBE):")
by_year_park[by_year_park.park_code == "SLBE"].head(10)


In [ ]:
# Q4: obvious duplicate geometries risking double-counting?
dup_wkb = glkn.geometry.apply(lambda g: g.wkb).duplicated(keep=False)
print(f"{dup_wkb.sum()} of {len(glkn)} confirmed polygons share an exact duplicate geometry -- negligible.")


**Q5 (Isle Royale custom HUC codes):** answered in Section 4 — effectively removes ISRO from any HUC10-level summary unless resolved separately; not a minor edge case.

**Q6 (MISS `false`-row HUC_12 anomaly):** checked directly against confirmed rows only — 0 of MISS's 4,623 confirmed rows have missing `HUC_12`. The anomaly was specific to MISS's rejected (`false`) rows, entirely outside this product's scope. Resolved, does not affect this product.

## B.5 Interpretation / Open Questions

### Measured (this notebook / `src/process_glkn.py`)

- Re-verified the full GLKN schema fresh against the prior inventory — total rows, true/false counts, and the 7-park set all matched exactly, with one real bug caught along the way (`change_occurred` stored as string `"true"`/`"false"`, not boolean).
- Geometry repair via `make_valid()` produced negligible area change (0.0% across all 7 parks) on confirmed disturbances, with invalidity rates (0.27%–4.43%) far lower than NCCN's.
- 35.78% of confirmed disturbance area cannot be assigned to a standard-derived HUC10, concentrated almost entirely in ISRO (99.9% of its confirmed rows).
- 1.43% of confirmed polygons have more than one agent; `uniqID` repeats such that polygon counts overstate distinct patch groups by roughly 15–16%.

### Documented (GLKN metadata + SLBE report)

- Disturbance polygons are human-interpreted LandTrendr candidates, not a wall-to-wall land-cover map (same distinction established for NCCN).
- False-positive (rejected) candidates are explicitly excluded from GLKN's own published summary analysis — consistent with this notebook's choice to exclude them from the primary product.
- GLKN's own SLBE report aggregates `HUC_12` to HUC10 for presentation, without stating that HUC10 is the underlying monitoring-extent boundary (the same caution applied to NCCN's HUC10/HUC12 test applies here — not yet tested for GLKN).

### 7.1 Isle Royale / Voyageurs non-standard HUC codes -- investigated, then paused

Checked GLKN_metadata.rtf, the Kirschbaum SLBE report, and the GDB's SCH_DATASET/SCH_RELEASE/SCH_UNIQUEID tables (confirmed these are pure ESRI internal schema/version-tracking tables, empty of any code definitions) -- none explain the non-standard codes.

**One clean, decisive finding from existing project data (not spatial inference):** every non-standard-code confirmed row, in both ISRO (16,023/16,023) and VOYA (926/926), has `loc_02=='canada'` -- a 100% correlation, zero exceptions either direction; all standard-code rows are `loc_02=='usa'`. `owner_type1` on the non-standard rows is dominated by "Crown Land Unpatented/Patent" (Canadian land-tenure terminology).

**This establishes *why* no standard code exists** -- the USGS Watershed Boundary Dataset (source of the standard 12-digit codes) does not cover Canadian territory -- **but not what the non-standard codes themselves specifically denote.** Investigation paused here per instruction, before completing spatial-coherence analysis or sourcing a Canadian boundary/code dataset.

**Resolved 2026-09-24, not by sourcing a Canadian HUC dataset, but by new data:** the authoritative `GLKN_LandTrendr_AOIs` per-park AOI (received 2026-09-24, see Appendix B.6) contains 99.99% of ISRO's and 100% of VOYA's existing attributed area, including their Canadian-portion (`loc_02=='canada'`) records -- this AOI appears to already have been drawn to include whatever generated those non-standard-HUC rows. Standard US HUC boundaries remain insufficient for ISRO/VOYA on their own, but that limitation is no longer load-bearing: the LandTrendr AOI is GLKN's authoritative analysis-area boundary, documented as context in B.1/Appendix B.6 (not used to clip Task 1's pixel counts).

### Interpretation / open questions (not yet settled)

- **Whether formal NPS park boundaries meaningfully under-represent GLKN's monitored extent**, the way they do for 3 of NCCN's 4 parks, has not yet been tested quantitatively (no inside/outside-boundary analysis was computed this pass). The maps in Section 5 make the *qualitative* pattern visible.
- **Resolved 2026-09-24**: yes -- `GLKN_LandTrendr_AOIs` is exactly this concept (see Appendix B.6), confirmed authoritative by its own embedded FGDC/Esri lineage, not inferred from area alone.

### Next steps (not started)

- Quantify GLKN reference-polygon area inside vs. outside the formal park boundary, per park (NCCN Step 5 equivalent) -- deferred.
- Per-park dissolved HUC-based analysis boundaries for GLKN -- superseded; the LandTrendr AOI (Appendix B.6) now serves this role and does not have ISRO/VOYA's Canadian-coverage gap.
- Move to ADS R6.

### B.6 Authoritative LandTrendr AOI investigation (added 2026-09-24)

`GLKN_LandTrendr_AOIs.shp` (received 2026-09-22, `data/raw/boundaries/glkn/`): 9 features (`park`, `Shape_Leng`, `Shape_Area`, `acres`) -- the 7 `park_code` values used throughout this notebook (APIS, INDU, ISRO, MISS, SACN, SLBE, VOYA), plus **PIRO and GRPO**, which have no reference disturbance data in this project's confirmed GLKN dataset. Areas are 1.9-330x each park's core NPS-unit size -- this is not a park-boundary layer.

**Confirmed authoritative, not inferred from area alone.** The shapefile's own embedded FGDC/Esri lineage records `{PARK}_LandTrendr_analysis_area` shapefiles being built and appended park-by-park, 2011 (APIS) through 2022 (SACN), into `LandTrendr_analysis_areas_by_park`, exported to the delivered file 2026-04-22. This is literally the AOI the GLKN LandTrendr change-detection analysis was run within, per park -- the boundary that generated the reference disturbance dataset in the first place, not a candidate boundary tested against it after the fact.

In [ ]:
aoi = pd.read_csv(QA_DIR / "nccn_glkn_aoi_total_pixel_counts.csv")
aoi = aoi[aoi["source"] == "GLKN"][["subregion", "aoi_pixel_count", "aoi_area_ha"]]

pixel_comp = pd.read_csv(QA_DIR / "nccn_glkn_aoi_constraint_comparison_subregion.csv")
pixel_comp = pixel_comp[pixel_comp["source"] == "GLKN-primary"]

table = aoi.merge(pixel_comp[["subregion", "existing_attributed_pixel_years", "pct_diff"]], on="subregion")
table["pct_attributed_pixel_years_in_aoi"] = (100 + table["pct_diff"].fillna(0)).round(3)
table = table.drop(columns=["pct_diff"]).set_index("subregion").reindex(GLKN_PARKS)
table


Every park is 99.99-100% contained; ISRO is the only park with any measurable difference (173 of 1,290,342 attributed pixel-years, 0.01%, per `outputs/qa/nccn_glkn_aoi_constraint_comparison_class.csv` almost entirely `blowdown`/`insect_disease_defo`). This is an essentially complete agreement between GLKN's own `park_code` assignment and its authoritative analysis-area AOI -- there is no meaningful boundary-generation question for GLKN the way there is for NCCN (Appendix A.6). **Task 1's pixel counts are not constrained to this AOI**; there is essentially nothing to constrain. Full method: `src/qa_new_aoi_containment.py` (vector-level containment), `src/qa_aoi_constrained_pixel_comparison.py` (pixel-level comparison, reused here from the shared boundary-AND-mask pattern in `rasterize_common.rasterize_ads_source`).

---
# Appendix C — ADS Region 6 / Region 10: Illustrative Polygon Samples

Illustrative random-polygon samples (not the full dataset) for individual-shape inspection, referenced from A.5/A.5-equivalent in each ADS source's Part A. The density maps in each source's own Part A (A.5) are the representative view of the full data.

### C.1 ADS Region 6 — Illustrative polygon sample (labeled, for shape inspection)

A random sample of individual polygons -- **not the full dataset** -- shown against ecoregion boundaries, so individual attributed shapes can be visually inspected. The density map above (A.5) is the representative view of the full data. *(Informal/illustrative QA visualization -- candidate for an appendix in the final report.)*

In [ ]:
SAMPLE_N = 3000
sample = ads_r6.sample(n=min(SAMPLE_N, len(ads_r6)), random_state=42)

fig, ax = plt.subplots(figsize=(10, 10))
viz.static_overlay_map([
    (ecoregions, dict(facecolor="none", edgecolor="black", linewidth=1.2), "EPA ecoregions"),
    (sample, dict(facecolor="#d73027", edgecolor="none", alpha=0.6), f"ADS R6 sample (n={len(sample):,} of {len(ads_r6):,})"),
], title="ADS R6 -- random polygon sample against ecoregion boundaries", ax=ax)
fig.tight_layout()


In [ ]:
# Interactive equivalent -- same sample (the full 913k-polygon dataset is not
# feasible to embed in an interactive map).
m = viz.interactive_overlay_map([
    (ecoregions, dict(color="black", weight=1.5, fillOpacity=0.0), "EPA ecoregions", True),
    (sample, viz.park_layer_style("reference"), f"ADS R6 sample (n={len(sample):,})", True),
], zoom_start=6)
m


In [ ]:
by_year = pd.read_csv(QA_DIR / "ads_r6_by_ecoregion_year.csv")
year_pivot = by_year.pivot(index="SURVEY_YEA", columns="us_l3code", values="polygon_count").fillna(0)

fig, ax = plt.subplots(figsize=(11, 5))
year_pivot.plot(ax=ax, linewidth=1.2)
ax.set_ylabel("polygon count")
ax.set_xlabel("survey year")
ax.set_title("ADS R6 attributed record count by year, per ecoregion (see the heatmap below for area instead of count)")
ax.legend(title="us_l3code", fontsize=7, ncol=2)
fig.tight_layout()


In [ ]:
by_dca = pd.read_csv(QA_DIR / "ads_r6_by_ecoregion_dca.csv")
print("Top 5 causal agents (DCA_COMMON) by area, per ecoregion:")
for code_, grp in by_dca.groupby("us_l3code"):
    top5 = grp.sort_values("area_m2", ascending=False).head(5)
    print(f"\n{code_}:")
    for r in top5.itertuples():
        print(f"  {r.DCA_COMMON}: {r.polygon_count:,} polygons, {r.area_m2/1e4:,.0f} ha")


### C.2 ADS Region 10 — Illustrative polygon sample (labeled, for shape inspection)

A random sample of individual polygons -- **not the full dataset** -- shown against HUC6 boundaries, so individual attributed shapes can be visually inspected. The density map above (A.5) is the representative view of the full data. *(Informal/illustrative QA visualization -- candidate for an appendix in the final report.)*

In [ ]:
SAMPLE_N = 3000
sample = ads_r10.sample(n=min(SAMPLE_N, len(ads_r10)), random_state=42)

fig, ax = plt.subplots(figsize=(10, 10))
viz.static_overlay_map([
    (huc6, dict(facecolor="none", edgecolor="black", linewidth=1.2), "HUC6 regions"),
    (sample, dict(facecolor="#d73027", edgecolor="none", alpha=0.6), f"ADS R10 sample (n={len(sample):,} of {len(ads_r10):,})"),
], title="ADS R10 -- random polygon sample against HUC6 boundaries", ax=ax)
fig.tight_layout()


In [ ]:
# Interactive equivalent -- same sample (the full 151k-polygon dataset is not
# feasible to embed in an interactive map).
m = viz.interactive_overlay_map([
    (huc6, dict(color="black", weight=1.5, fillOpacity=0.0), "HUC6 regions", True),
    (sample, viz.park_layer_style("reference"), f"ADS R10 sample (n={len(sample):,})", True),
], zoom_start=5)
m


In [ ]:
by_year = pd.read_csv(QA_DIR / "ads_r10_by_huc6_year.csv")
year_pivot = by_year.pivot(index="SURVEY_YEAR", columns="huc6_code", values="record_count").fillna(0)

fig, ax = plt.subplots(figsize=(11, 5))
year_pivot.plot(ax=ax, linewidth=1.2, legend=False)
ax.set_ylabel("record count")
ax.set_xlabel("survey year")
ax.set_title("ADS R10 attributed record count by year, per HUC6 (legend omitted -- 20 series; see the heatmap below for a legible per-region view)")
fig.tight_layout()


In [ ]:
by_dca = pd.read_csv(QA_DIR / "ads_r10_by_huc6_dca.csv")
print("Top 5 causal agents (DCA_COMMON_NAME) by area, per HUC6:")
for code_, grp in by_dca.groupby("huc6_code"):
    top5 = grp.sort_values("area_m2", ascending=False).head(5)
    print(f"\n{code_}:")
    for r in top5.itertuples():
        print(f"  {r.DCA_COMMON_NAME}: {r.record_count:,} records, {r.area_m2/1e4:,.0f} ha")
